<a href="https://colab.research.google.com/github/FrantisekBrandejs/brandejs_BP_26_RQA/blob/main/brandejs_BP_26_RQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **VYUŽITÍ REKURENTNÍ KVANTIFIKAČNÍ ANALÝZY PRO INTERPRETACI OČNÍCH POHYBŮ PŘI ČTENÍ STATICKÝCH MAP** | bakalářská práce | František BRANDEJS, 2026

In [ ]:
# @title Import knihoven { display-mode: "form" }
# @markdown Spusťte tuto buňku pro import potřebných knihoven a nastavení Colab prostředí. <br>
# @markdown Použité knihovny: google.colab, os, pandas, numpy, matplotlib.pyplot, ipywidgets, seaborn, warnings, csv, time, re, math, aj.
#Knihovny
from google.colab import drive
import os
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
from matplotlib.ticker import MaxNLocator
import matplotlib.patheffects as path_effects
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu
import warnings
import csv
import time
import re
import math
from scipy.stats import linregress
warnings.filterwarnings('ignore')
print("Knihovny načteny.")
print("Můžete spustit další blok kódu")

In [ ]:
# @title Nastavení cest k souborům { display-mode: "form" }
# @markdown Vyplňte cesty k vašim vstupním datům.
# @markdown Skript vyžaduje 2 vstupní soubory: <br>
# @markdown 1. Segmentovaná data (export pomocí nástroje GazePlotter, formát .csv)
# @markdown 2. Tabulka respondentů (např. .csv nebo .tsv). Měla by mít dva sloupce: 1. jméno respondenta, 2. jeho zařazení (např. Nováček/Expert).

# --- VSTUPNÍ DATA ---
PROJECT_FOLDER = "drive/MyDrive/BAK-RQA/" # @param {"type":"string"}
FILE_NAME_SEGMENTED = "GazePlotter-SegmentedData.csv" # @param {type:"string"}
FILE_NAME_PARTICIPANTS = "participant_table.tsv" # @param {type:"string"}

# @markdown ---
# @markdown **Parametry analýzy (RQA)** minimální délky linií v RP
RQA_L_MIN = 2 # @param {type:"integer"}
RQA_V_MIN = 2 # @param {type:"integer"}


if not PROJECT_FOLDER.endswith('/'):
    PROJECT_FOLDER += '/'

FILE_EYETRACKING_CSV = f'{PROJECT_FOLDER}{FILE_NAME_SEGMENTED}'
FILE_PARTICIPANTS = f'{PROJECT_FOLDER}{FILE_NAME_PARTICIPANTS}'

FILE_SingleAOI_CSV = f'{PROJECT_FOLDER}SegmentedData-SingleAOI.csv'
FILE_COMPLETE_CSV = f'{PROJECT_FOLDER}SegmentedData-complete.csv'
FILE_OUT_AOI = f'{PROJECT_FOLDER}rqa_results_AOI.csv'
FILE_OUT_XY = f'{PROJECT_FOLDER}rqa_results_XY.csv'

if not os.path.exists(FILE_EYETRACKING_CSV):
    print(f"POZOR: Segmentovaná data {FILE_EYETRACKING_CSV} nebyla nalezena!")
elif not os.path.exists(FILE_PARTICIPANTS):
    print(f"POZOR: Tabulka respondentů {FILE_PARTICIPANTS} nebyla nalezena!")
else:
    print("Vstupní data nalezena! Můžete pokračovat na další krok.")

EXPERIENCE_COLORS = {}

In [ ]:
# @title Výběr stimulů a vyloučení nežádoucích respondentů { display-mode: "form" }
# @markdown V tomto kroku máte možnost upřesnit, jaká data vstoupí do analýzy. Po spuštění buňky:
# @markdown * V levém seznamu vyberte konkrétní stimuly (mapy), které chcete analyzovat.
# @markdown * V pravém seznamu můžete označit respondenty, které chcete z analýzy zcela vyřadit.
# @markdown * Výběr potvrďte kliknutím na tlačítko níže, čímž se data uloží pro další kroky.

# 1. Načtení jmen z tabulky
try:
    if FILE_PARTICIPANTS.endswith('.tsv'):
        respondent_table = pd.read_csv(FILE_PARTICIPANTS, sep='\t')
    else:
        respondent_table = pd.read_csv(FILE_PARTICIPANTS, sep=None, engine='python')
    COL_PARTICIPANT = respondent_table.columns[0]
    COL_EXPERIENCE = respondent_table.columns[1]
    all_participants = sorted(respondent_table[COL_PARTICIPANT].dropna().astype(str).unique().tolist())
except Exception as e:
    all_participants = []
    print(f"Chyba při čtení tabulky respondentů: {e}")

# 2. Načtení stimulů ze segmentovaných dat
try:
    df_stimuli = pd.read_csv(FILE_EYETRACKING_CSV, usecols=['stimulus'], low_memory=False)
    all_stimuli = sorted(df_stimuli['stimulus'].dropna().unique().tolist())
except Exception as e:
    all_stimuli = []
    print(f"Nelze přečíst stimuly: {e}")

predvyber_stimulu = all_stimuli

# 3. Vykreslení UI
w_stimuli = widgets.SelectMultiple(
    options=all_stimuli, value=predvyber_stimulu,
    description='Vyberte mapy k analýze:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px', height='150px')
)

w_exclude = widgets.SelectMultiple(
    options=all_participants, value=[],
    description='Vyloučit tyto respondenty:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px', height='150px')
)
lbl_exclude = widgets.Label("(Naměřené fixace vybraných participantů budou smazány. Pro výběr více participantů držte CTRL.)", style={'color': 'gray', 'font_size': '11px'})

btn_save_krok2 = widgets.Button(description='Uložit výběr stimulů a participantů', button_style='primary', layout=widgets.Layout(width='200px', margin='10px 0 0 0'))
out_krok2 = widgets.Output()

def on_krok2_save(b):
    global SELECTED_STIMULI, EXCLUDED_PARTICIPANTS
    with out_krok2:
        clear_output(wait=True)
        SELECTED_STIMULI = list(w_stimuli.value)
        EXCLUDED_PARTICIPANTS = list(w_exclude.value)

        print("ÚSPĚŠNĚ ULOŽENO DO PAMĚTI:")
        print(f"  • Budeme analyzováno {len(SELECTED_STIMULI)} map/stimulů.")
        if EXCLUDED_PARTICIPANTS:
            print(f"  • Z analýzy vymazáno {len(EXCLUDED_PARTICIPANTS)} nežádoucích respondentů.")
        else:
            print("  • Nebyl vyloučen žádný respondent.")
        print("\n Nyní je možno spustit další buňku.")

btn_save_krok2.on_click(on_krok2_save)

display(widgets.HBox([w_stimuli, widgets.VBox([w_exclude, lbl_exclude])]))
display(btn_save_krok2, out_krok2)

In [ ]:
# @title Sloučení dat a úprava AOI { display-mode: "form" }
# @markdown V tomto kroku sjednotíte názvosloví oblastí zájmu (AOI) a sloučíte oční data s tabulkou respondentů. Po spuštění buňky:
# @markdown * Nejprve klikněte na tlačítko pro zobrazení nalezených AOI. Skript analyzuje vybrané stimuly a vypíše všechny detekované oblasti zájmu.
# @markdown * V zobrazeném seznamu můžete nepřehledné názvy z Gazeplotteru přejmenovat a zařadit je do příslušné makro-kategorie (Mapa, Legenda, nebo Ostatní).
# @markdown * Následně data zpracujte druhým tlačítkem. Skript aplikuje vaše nová AOI, vyřadí dříve vyloučené respondenty, spáruje oční data s tabulkou zkušeností a vytvoří finální datovou sadu pro výpočty.
if 'SELECTED_STIMULI' not in globals():
    print("Nejprve musíte spustit a uložit předchozí krok")
else:
    DEFAULT_AOI_NAMES = {
        'nan': 'Miss', 'All': 'Others', '01;All': '1', '02;All': '2', '03;All': '3',
        '04;All': '4', '05;All': '5', '06;All': '6', '07;All': '7', '08;All': '8', '09;All': '9',
        '01;02;All' : '1', '02;03;All' : '2', '03;04;All' : '3', '04;05;All' : '4',
        '05;06;All' : '5', '06;07;All' : '6', '07;08;All' : '7', '08;09;All' : '8',
        'All;Title': 'Title', 'All;Imprint': 'Imprint', 'All;Brightest': 'Brightest',
        'All;Darkest': 'Darkest', 'All;Target': 'Target', 'All;target': 'Target',
        'All;Brightest;Target' : 'Target', 'All;Darkest;Target' : 'Target',
        'All;Darkest;target' : 'Target',
    }
    DEFAULT_CAT_MAPA = ['Others']
    DEFAULT_CAT_LEGENDA = ['1', '2', '3', '4', '5', '6', '7', '8', '9']
    # Mapování písmen v názvu stimulu na počet intervalů legendy.
    # Specifické pro dataset ve formátu R{číslo}{písmeno} (např. R3a).
    # Pro jiný formát stimulů tato část nefunguje – Target_Interval bude prázdný.
    INTERVAL_KEY = {'a': '2', 'b': '3', 'c': '4', 'd': '5', 'e': '6', 'f': '7', 'g': '8'}

    btn_load_aoi = widgets.Button(description='1. Zobrazit nalezené AOI', button_style='info', layout=widgets.Layout(width='200px'))
    out_aoi_list = widgets.Output()
    btn_process = widgets.Button(description='2. Zpracovat a uložit sloučená data', button_style='success', layout=widgets.Layout(display='none', margin='15px 0px 0px 0px', width='300px'))
    out_process = widgets.Output()
    aoi_widgets = {}

    def on_load_aoi_clicked(b):
        with out_aoi_list:
            clear_output(wait=True); aoi_widgets.clear()

            print("Načítám data...")
            df_temp = pd.read_csv(FILE_EYETRACKING_CSV, usecols=['stimulus', 'AOI', 'eyemovementtype'], low_memory=False)
            df_temp = df_temp[(df_temp['eyemovementtype'] == 0) & (df_temp['stimulus'].isin(SELECTED_STIMULI))]
            unique_aois = sorted(df_temp['AOI'].dropna().unique().tolist())

            vbox_items = [widgets.HBox([widgets.Label("Původní AOI z Gazeplotteru:", layout={'width':'250px'}),
                                        widgets.Label("Nový název:", layout={'width':'150px'}),
                                        widgets.Label("Mapa/Legenda/Ostatní:", layout={'width':'150px'})])]

            for orig_aoi in unique_aois:
                orig_str = str(orig_aoi)
                new_n = DEFAULT_AOI_NAMES.get(orig_str, orig_str)
                cat_def = 'Mapa (0)' if new_n in DEFAULT_CAT_MAPA else ('Legenda (1)' if new_n in DEFAULT_CAT_LEGENDA else 'Ostatni (2)')
                w_n = widgets.Text(value=new_n, layout={'width':'140px'})
                w_c = widgets.Dropdown(options=['Mapa (0)', 'Legenda (1)', 'Ostatni (2)'], value=cat_def, layout={'width':'160px'})
                aoi_widgets[orig_str] = {'name': w_n, 'cat': w_c}
                vbox_items.append(widgets.HBox([widgets.Label(value=orig_str, layout={'width':'250px'}), w_n, w_c]))

            display(widgets.VBox(vbox_items))
            btn_process.layout.display = 'block'

    def on_process_clicked(b2):
        global df_complete
        with out_process:
            clear_output(wait=True)
            print("Filtruji, překládám AOI a slučuji s tabulkou respondentů...")

            try:
                df = pd.read_csv(FILE_EYETRACKING_CSV, low_memory=False)
                df_fix = df[(df['eyemovementtype'] == 0) & (df['stimulus'].isin(SELECTED_STIMULI))].copy()

                if df_fix.empty:
                    print("Chyba: Pro vybrané stimuly nebyly nalezeny žádné fixace.")
                    return

                # Bezpečné získání cílového intervalu (odolné vůči cizím názvům)
                try:
                    df_fix['target_letter'] = df_fix['stimulus'].astype(str).str.extract(r'^R\d+([a-gA-G])', expand=False).str.lower()
                    df_fix['Target_Interval'] = df_fix['target_letter'].map(INTERVAL_KEY)
                    df_fix = df_fix.drop(columns=['target_letter'])
                except Exception:
                    df_fix['Target_Interval'] = None

                # Očištění jmen fixací
                df_fix['participant'] = df_fix['participant'].astype(str).str.strip()
                df_fix['clean_name'] = df_fix['participant'].str.replace(r'^Recording\d+\s+', '', regex=True)
                df_fix.loc[df_fix['clean_name'] == '', 'clean_name'] = df_fix['participant']

                df_fix = df_fix.dropna(subset=['AOI']).drop(columns=['eyemovementtype'])

                # Aplikace AOI
                final_mapping = {orig: w['name'].value.strip() for orig, w in aoi_widgets.items()}
                final_categories = {}
                for orig, w in aoi_widgets.items():
                    name = w['name'].value.strip()
                    cat = w['cat'].value
                    if 'Mapa' in cat: final_categories[name] = '0'
                    elif 'Legenda' in cat: final_categories[name] = '1'
                    else: final_categories[name] = '2'

                df_fix['AOI Single'] = df_fix['AOI'].astype(str).map(lambda x: final_mapping.get(x, x))
                df_fix['AOI Aggregated'] = df_fix['AOI Single'].apply(lambda x: final_categories.get(str(x), '2'))

                # Zpracování tabulky respondentů (Bezpečný LEFT JOIN místo INNER JOIN)
                if 'respondent_table' in globals() and not respondent_table.empty and COL_PARTICIPANT in respondent_table.columns:
                    respondent_table_filtered = respondent_table.copy()
                    if EXCLUDED_PARTICIPANTS:
                        respondent_table_filtered = respondent_table_filtered[~respondent_table_filtered[COL_PARTICIPANT].isin(EXCLUDED_PARTICIPANTS)]

                    respondent_table_filtered['join_name'] = respondent_table_filtered[COL_PARTICIPANT].str.replace(r'^Recording\d+\s+', '', regex=True)
                    respondent_table_filtered.loc[respondent_table_filtered['join_name'] == '', 'join_name'] = respondent_table_filtered[COL_PARTICIPANT]

                    # Finální Merge (LEFT JOIN)
                    df_complete = pd.merge(df_fix, respondent_table_filtered, left_on='clean_name', right_on='join_name', how='left')

                    if COL_EXPERIENCE in df_complete.columns:
                        df_complete = df_complete.rename(columns={COL_EXPERIENCE: 'Zkusenosti'})
                        df_complete['Zkusenosti'] = df_complete['Zkusenosti'].fillna('Nezadáno')
                    else:
                        df_complete['Zkusenosti'] = 'Nezadáno'

                    df_complete = df_complete.drop(columns=['clean_name', 'join_name', COL_PARTICIPANT], errors='ignore')
                else:
                    # Fallback postup pro případ, že tabulka zkušeností chybí
                    df_complete = df_fix.copy()
                    df_complete['Zkusenosti'] = 'Nezadáno'
                    df_complete = df_complete.drop(columns=['clean_name'], errors='ignore')

                df_complete = df_complete.loc[:, ~df_complete.columns.duplicated()]

                # Uložení
                df_complete.to_csv(FILE_SingleAOI_CSV, index=False)

                print("Zpracování a sloučení úspěšně dokončeno!")
                print(f"Do dalších analýz jde: {df_complete['participant'].nunique()} platných respondentů.")
                print(f"Zjištěné kategorie zkušeností: {', '.join(map(str, df_complete['Zkusenosti'].unique().tolist()))}")
                print(f"Data uložena jako: {FILE_SingleAOI_CSV}")

            except Exception as e:
                print(f"\nNastala chyba při zpracování: {e}")

    btn_load_aoi.on_click(on_load_aoi_clicked)
    btn_process.on_click(on_process_clicked)

    display(btn_load_aoi, out_aoi_list, btn_process, out_process)

In [ ]:
# @title Filtrace odlehlých hodnot (Outlierů) pomocí Mezikvartilového rozpětí { display-mode: "form" }
# @markdown Tento krok slouží k očištění dat od extrémních hodnot, které by mohly zkreslit výsledky analýzy.
# @markdown * Skript vypočítá statistické hranice (mezikvartilové rozpětí) pro počet fixací u každé mapy zvlášť.
# @markdown * Následně automaticky odstraní záznamy těch respondentů, kteří se vymykají běžnému chování (mají extrémně málo, nebo naopak extrémně mnoho fixací).
# @markdown * Algoritmus rovněž hlídá matematické minimum fixací nutných pro výpočet RQA, které je stanoveno jako minimální stanovená délka linie + 1. Minimální délku je možné upravit v kroku nastavení cest k souborům
# @markdown * Kliknutím na tlačítko provedete filtraci a připravíte data pro finální výpočet metrik.

try:
    df_complete = pd.read_csv(FILE_SingleAOI_CSV)
    sequence_lengths = df_complete.groupby(['participant', 'stimulus', 'Zkusenosti']).size().reset_index(name='fixation_count')
except Exception as e:
    print("Chyba: Soubor s daty nebyl nalezen. Spusťte nejprve předchozí kroky.")
    df_complete = pd.DataFrame()
    sequence_lengths = pd.DataFrame()

lbl_filter = widgets.Label("Filtrace extrémních hodnot (mezikvartilové rozpětí - IQR)", layout=widgets.Layout(font_weight='bold', font_size='14px'))
btn_filter = widgets.Button(description='Provést odstranění outlierů', button_style='warning', layout=widgets.Layout(width='250px'))
out_filter = widgets.Output()

def on_filter_clicked(b):
    with out_filter:
        clear_output(wait=True)
        if df_complete.empty: return

        print("Počítám statistické hranice pro každý stimul zvlášť...")

        # Minimální délka sekvence pro to, aby vůbec šla spočítat RQA
        # (pokud je minimální délka linie 2, musí být minimálně 3 fixace)
        MATH_MINIMUM = RQA_L_MIN + 1 if 'RQA_L_MIN' in globals() else 3

        def get_outlier_bounds(group):
            # 1. Výpočet kvartilů
            q1 = group['fixation_count'].quantile(0.25)
            q3 = group['fixation_count'].quantile(0.75)

            # 2. Mezikvartilové rozpětí
            iqr = q3 - q1

            # 3. Výpočet spodní a horní hranice
            # Spodní hranice nesmí klesnout pod matematické minimum pro výpočet RQA
            lower_bound = max(q1 - 1.5 * iqr, MATH_MINIMUM)
            upper_bound = q3 + 1.5 * iqr

            return pd.Series({'lower_bound': lower_bound, 'upper_bound': upper_bound})

        # Aplikace výpočtu pro každý stimul izolovaně
        group_bounds = sequence_lengths.groupby(['stimulus']).apply(get_outlier_bounds).reset_index()
        merged_lengths = sequence_lengths.merge(group_bounds, on=['stimulus'])

        # Filtrace respondentů, kteří se vešli do hranic
        valid_sequences = merged_lengths[
            (merged_lengths['fixation_count'] >= merged_lengths['lower_bound']) &
            (merged_lengths['fixation_count'] <= merged_lengths['upper_bound'])
        ]

        removed_count = len(merged_lengths) - len(valid_sequences)

        print("-" * 50)
        print(f"Celkový počet sekvencí před filtrací: {len(merged_lengths)}")
        print(f"Počet odstraněných sekvencí (odlehlých hodnot): {removed_count}")
        print(f"Počet platných sekvencí po filtraci: {len(valid_sequences)}")
        print("-" * 50)

        # Uložení pouze těch dat, která prošla sítem
        df_cleaned = df_complete.merge(valid_sequences[['participant', 'stimulus']], on=['participant', 'stimulus'], how='inner')
        df_cleaned.to_csv(FILE_COMPLETE_CSV, index=False)

        print(f"Očištěná data byla úspěšně uložena.")
        print("Nyní můžete přejít k výpočtu RQA metrik.")

btn_filter.on_click(on_filter_clicked)

if not df_complete.empty:
    display(widgets.VBox([lbl_filter, btn_filter, out_filter]))

In [ ]:
# @title Výpočet rádiusu pro RQA (XY přístup) { display-mode: "form" }
# @markdown Tento krok slouží k matematickému výpočtu biologického rádiusu pro prostorovou analýzu RQA (přístup XY).
# @markdown * Vyplňte parametry vašeho experimentálního uspořádání (rozlišení monitoru, úhlopříčka a průměrná vzdálenost očí respondenta od obrazovky).
# @markdown * Skript na základě těchto fyzických parametrů vypočítá přesnou velikost jednoho pixelu a pomocí goniometrie určí velikost zorného pole (fovey) přímo v pixelech.
# @markdown * Vypočtená hodnota se automaticky uloží do paměti Colabu a bude použita jako maximální povolená vzdálenost pro detekci fixací ve stejném bodě během výpočtu RQA metrik.
# --- UŽIVATELSKÉ ROZHRANÍ ---
lbl_title = widgets.Label("Parametry eye-trackingového experimentu:", layout=widgets.Layout(font_weight='bold', font_size='14px'))

# Vstupy od uživatele (předvyplněno pro typický 24" Full HD monitor)
w_res_x = widgets.IntText(value=1920, description='Rozlišení X (px):', style={'description_width': 'initial'})
w_res_y = widgets.IntText(value=1080, description='Rozlišení Y (px):', style={'description_width': 'initial'})
w_diagonal = widgets.FloatText(value=24.0, description='Úhlopříčka (palce):', style={'description_width': 'initial'})
w_distance = widgets.FloatText(value=80.0, description='Vzdálenost očí (cm):', style={'description_width': 'initial'})
w_angle = widgets.FloatText(value=2, step=0.1, description='Zorný úhel fovey (°):', style={'description_width': 'initial'})

btn_calc = widgets.Button(description='Vypočítat a uložit do paměti', button_style='primary', layout=widgets.Layout(width='250px', margin='15px 0 0 0'))
out_calc = widgets.Output()

def calculate_radius(b):
    with out_calc:
        clear_output(wait=True)

        res_x = w_res_x.value
        res_y = w_res_y.value
        diag_inch = w_diagonal.value
        dist_cm = w_distance.value
        angle_deg = w_angle.value

        # 1. Výpočet hustoty pixelů (PPI - Pixels Per Inch)
        diag_px = math.sqrt(res_x**2 + res_y**2)
        ppi = diag_px / diag_inch

        # 2. Fyzická velikost jednoho pixelu v cm (1 palec = 2.54 cm)
        pixel_size_cm = 2.54 / ppi

        # 3. Výpočet poloměru fovey na obrazovce v cm
        # Polovina zorného úhlu
        half_angle_rad = math.radians(angle_deg / 2)
        radius_cm = dist_cm * math.tan(half_angle_rad)

        # 4. Převod fyzického poloměru na pixely
        radius_px = int(round(radius_cm / pixel_size_cm))

        # --- ULOŽENÍ DO GLOBÁLNÍ PROMĚNNÉ ---
        global CALCULATED_RQA_RADIUS_PX
        CALCULATED_RQA_RADIUS_PX = radius_px

        print("ÚSPĚŠNĚ VYPOČTENO A ULOŽENO!")
        print("-" * 40)
        print(f"Hustota monitoru: {ppi:.1f} PPI")
        print(f"Velikost 1 pixelu: {pixel_size_cm:.4f} cm")
        print(f"Poloměr fovey na monitoru: {radius_cm:.2f} cm")
        print("-" * 40)
        print(f"Výsledný RQA Rádius: {radius_px} pixelů")
        print("-" * 40)
        print("Tuto hodnotu si skript zapamatoval. Až spustíte hlavní výpočet RQA (XY), automaticky si ji z paměti vytáhne.")

btn_calc.on_click(calculate_radius)

ui = widgets.VBox([
    lbl_title,
    widgets.HBox([w_res_x, w_res_y]),
    widgets.HBox([w_diagonal, w_distance]),
    w_angle,
    btn_calc
])
display(widgets.VBox([ui, out_calc]))

In [ ]:
# @title Výpočet RQA metrik (AOI i XY přístup) { display-mode: "form" }
# @markdown Tento krok představuje výpočetní jádro celého skriptu. Z připravených očních pohybů zde budou matematicky extrahovány finální kognitivní metriky (Recurrence Rate, Determinismus a Laminarita).
# @markdown * **Vstupní data:** Z roletového menu vyberte datovou sadu k analýze. Pro nejrelevantnější výsledky se důrazně doporučuje ponechat výchozí "Očištěná data", která již byla zbavena statistických odchylek a zajistí tak nezkreslené výsledky. Pokud k odstranění outlierů vaše data nepotřebují tento skript, stačí nastavit druhou možnost.
# @markdown * **Nastavení rádiusu:** Posuvník určuje maximální povolenou vzdálenost dvou fixací (v pixelech), aby byly algoritmem vyhodnoceny jako návrat na stejné místo. Pokud jste spustili předchozí krok (Výpočet rádiusu), hodnota se automaticky přednastaví na váš výpočet.
# @markdown * **Metodika výpočtu:** Skript zcela autonomně provede dva paralelní výpočty. AOI přístup zkonstruuje vzdálenostní matici na základě textové shody pojmenovaných oblastí. XY přístup naopak využije euklidovskou geometrii a zadaný rádius.
# @markdown * **Výstupy:** Výsledné zprůměrované hodnoty pro každého respondenta a stimul budou exportovány do dvou samostatných CSV souborů ve vaší pracovní složce. Tyto soubory následně poslouží pro veškeré grafické vizualizace.
# --- 1. NAČTENÍ CEST A PAMĚTI ---
# Definujeme cesty pro případ, že uživatel přeskočil první buňky
try:
    _folder = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    _cleaned = FILE_COMPLETE_CSV if 'FILE_COMPLETE_CSV' in globals() else _folder + 'SegmentedData-cleaned.csv'
    _complete = FILE_SingleAOI_CSV if 'FILE_SingleAOI_CSV' in globals() else _folder + 'SegmentedData-complete.csv'
    _raw = FILE_EYETRACKING_CSV if 'FILE_EYETRACKING_CSV' in globals() else _folder + 'GazePlotter-SegmentedData.csv'
except:
    _folder, _cleaned, _complete, _raw = 'drive/MyDrive/BAK-RQA/', '', '', ''

# Mapování pro výběr v UI
FILE_OPTIONS = {
    'Očištěná data (Doporučeno - po filtraci outlierů)': _cleaned,
    'Kompletní data (Sloučená, ale bez filtrace)': _complete,
    'Surová data (Přímý export z GazePlotteru)': _raw
}

# Kontrola rádiusu z paměti (předchozí buňka kódu)
if 'CALCULATED_RQA_RADIUS_PX' in globals():
    default_radius = CALCULATED_RQA_RADIUS_PX
    radius_status = f"✅ Načten rádius: {default_radius} px"
    status_color = "green"
else:
    default_radius = 50
    radius_status = "Používáte manuální nastavení rádiusu (výpočet z Kroku 4 nenalezen)."
    status_color = "gray"

# --- 2. UŽIVATELSKÉ ROZHRANÍ ---
lbl_main = widgets.Label("Nastavení nezávislého výpočtu RQA:", layout=widgets.Layout(font_weight='bold', font_size='15px'))

dd_source = widgets.Dropdown(
    options=list(FILE_OPTIONS.keys()),
    value=list(FILE_OPTIONS.keys())[0],
    description='Vstupní soubor:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

radius_slider = widgets.IntSlider(
    value=default_radius, min=10, max=300, step=1,
    description='RQA Radius (px):',
    style={'description_width': 'initial'}
)
lbl_radius_info = widgets.Label(radius_status, style={'text_color': status_color})

btn_calc = widgets.Button(description='Spustit komplexní analýzu', button_style='success', layout=widgets.Layout(width='250px', margin='15px 0 0 0'))
out_calc = widgets.Output()

# --- 3. MATEMATICKÉ JÁDRO ---
def get_line_lengths(arr, min_length=2):
    pad = np.pad(arr, (1, 1), mode='constant', constant_values=0)
    diffs = np.diff(pad)
    starts = np.where(diffs == 1)[0]
    ends = np.where(diffs == -1)[0]
    lengths = ends - starts
    return lengths[lengths >= min_length]

def extract_rqa_from_rp(RP, N, l_min, v_min):
    recurrent_points = np.sum(RP)
    total_possible = N * (N - 1)
    RR = recurrent_points / total_possible if total_possible > 0 else 0
    diag_lines = []
    for d in range(1, N):
        diag = np.diagonal(RP, offset=d)
        diag_lines.extend(get_line_lengths(diag, l_min))
    DET = np.sum(diag_lines) / (recurrent_points / 2) if recurrent_points > 0 and diag_lines else 0
    vert_lines = []
    for c in range(N):
        vert_lines.extend(get_line_lengths(RP[:, c], v_min))
    LAM = np.sum(vert_lines) / recurrent_points if recurrent_points > 0 and vert_lines else 0
    return pd.Series({'RR': RR, 'DET': DET, 'LAM': LAM})

# --- 4. LOGIKA VÝPOČTU ---
def on_calc_clicked(b):
    with out_calc:
        clear_output(wait=True)
        file_path = FILE_OPTIONS[dd_source.value]

        if not os.path.exists(file_path):
            print(f"CHYBA: Soubor nebyl nalezen na cestě: {file_path}")
            print("Zkontrolujte, zda jste soubor nezapomněli vygenerovat nebo zda je správně cesta v Kroku 1.")
            return

        print(f"Zahajuji výpočet z: {dd_source.value}...")
        try:
            df = pd.read_csv(file_path, low_memory=False)
            # Zajištění existence sloupců pro různé typy vstupů
            if 'participant' not in df.columns: df['participant'] = 'Unknown'
            if 'Zkusenosti' not in df.columns: df['Zkusenosti'] = 'Undefined'

            # AOI VÝPOČET
            if 'AOI Single' in df.columns:
                print("1/2 Počítám Sémantickou RQA (AOI)...")
                df_aoi = df.dropna(subset=['AOI Single'])
                res_aoi = df_aoi.groupby(['participant', 'stimulus', 'Zkusenosti']).apply(
                    lambda g: extract_rqa_from_rp((g['AOI Single'].values[:, None] == g['AOI Single'].values[None, :]).astype(int), len(g), 2, 2)
                ).reset_index()
                res_aoi.to_csv(_folder + "rqa_results_AOI.csv", index=False)

            # XY VÝPOČET
            if 'x' in df.columns and 'y' in df.columns:
                print(f"2/2 Počítám Prostorovou RQA (XY, Radius={radius_slider.value}px)...")
                df_xy = df.dropna(subset=['x', 'y'])
                def run_spatial(g):
                    coords = np.column_stack((g['x'], g['y']))
                    RP = (cdist(coords, coords) <= radius_slider.value).astype(int)
                    np.fill_diagonal(RP, 0)
                    return extract_rqa_from_rp(RP, len(g), 2, 2)

                res_xy = df_xy.groupby(['participant', 'stimulus', 'Zkusenosti']).apply(run_spatial).reset_index()
                res_xy['Radius_px'] = radius_slider.value
                res_xy.to_csv(_folder + "rqa_results_XY.csv", index=False)

            print("\n HOTOVO! Výsledky byly uloženy do Vaší složky na Drive.")
            print(f"Vstupy pro grafy: rqa_results_AOI.csv a rqa_results_XY.csv")

        except Exception as e:
            print(f"Nastala chyba při výpočtu: {e}")

btn_calc.on_click(on_calc_clicked)
display(widgets.VBox([lbl_main, dd_source, lbl_radius_info, radius_slider, btn_calc, out_calc]))

In [ ]:
# @title Recurrence plot, Boxplot, Scatter plot, Radar plot { display-mode: "form" }
# @markdown Tento krok představuje vizualizační panel. Zde se vypočtená data transformují do grafických výstupů, které umožňují interpretaci kognitivních strategií.
# @markdown * Z roletového menu zcela nahoře si zvolte, zda chcete vizualizovat výsledky AOI nebo XY přístupu.
# @markdown * Nástroj je logicky rozdělen do čtyř záložek, mezi kterými můžete přepínat:
# @markdown * **1. Recurrence plot (RP):** Matice pro hlubokou kvalitativní analýzu jednotlivce. Ukazuje chronologický vývoj fixací konkrétního respondenta nad vybraným stimulem. Zvýrazněné body v grafu exaktně ukazují okamžiky, kdy se zrak vrátil na stejné místo nebo do stejné sémantické kategorie. Barvy fixací v matici jsou nastaveny dle samotných odstínů jednitlivých intervalů legendy. Zelený pruh označuje fixaci, která připadla intervalu legendy odpovídající odstínu určeného v mapovém poli.
# @markdown * **2. Boxplot:** Kvantitativní a statistické zhodnocení. Zobrazuje krabicové grafy s rozdělením hodnot hlavních metrik (RR, DET, LAM) a porovnává je mezi zkušenostními skupinami. V grafech jsou zaneseny i přesné hodnoty mediánů.
# @markdown * **3. Scatter plot:** Analýza kognitivních strategií. Znáorňuje systémovost zraku (Determinismus na ose X) vůči míře lokálního zasekávání (Laminarita na ose Y). Velikost jednotlivých bodů odpovídá celkové míře vracení se (Recurrence Rate). Lze analyzovat surová data i agregované průměry.
# @markdown * **4. Radar plot:** Holistický profil. Vykresluje průměrné hodnoty všech tří metrik současně na pavučinovém grafu, což umožňuje rychlé vizuální srovnání celkového chování mezi experty a nováčky.

# --- 1. NAČTĚNÍ DAT ---
try:
    folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    file_complete = FILE_COMPLETE_CSV if 'FILE_COMPLETE_CSV' in globals() else folder_path + 'SegmentedData-cleaned.csv'
    file_rqa_aoi = FILE_OUT_AOI if 'FILE_OUT_AOI' in globals() else folder_path + 'rqa_results_AOI.csv'
    file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
except NameError:
    folder_path = 'drive/MyDrive/BAK-RQA/'
    file_complete = folder_path + 'SegmentedData-cleaned.csv'
    file_rqa_aoi = folder_path + 'rqa_results_AOI.csv'
    file_rqa_xy = folder_path + 'rqa_results_XY.csv'

try:
    df_complete = pd.read_csv(file_complete, low_memory=False)
except Exception as e:
    print("Data nebyla nalezena. Zkontrolujte cestu k souboru SegmentedData-cleaned.csv.")
    raise

df_rqa_aoi = pd.read_csv(file_rqa_aoi) if os.path.exists(file_rqa_aoi) else pd.DataFrame()
df_rqa_xy = pd.read_csv(file_rqa_xy) if os.path.exists(file_rqa_xy) else pd.DataFrame()

options_mode = []
if not df_rqa_aoi.empty: options_mode.append('AOI')
if not df_rqa_xy.empty: options_mode.append('XY')

if not options_mode:
    print("Nebyla nalezena RQA data. Spusťte předchozí výpočtové kroky.")
else:
    dd_rqa_mode = widgets.Dropdown(options=options_mode, description='Režim RQA:', style={'description_width': 'initial'})

    def get_active_data():
        return df_rqa_xy.copy() if 'XY' in dd_rqa_mode.value else df_rqa_aoi.copy()

    if 'EXPERIENCE_COLORS' not in globals():
        EXPERIENCE_COLORS = {'Nováček': '#ff7f0e', 'Expert': '#1f77b4'}

    # Přiřazení barev dynamicky pro cizí skupiny
    combined_groups = set()
    if not df_rqa_aoi.empty: combined_groups.update(df_rqa_aoi['Zkusenosti'].dropna().unique())
    if not df_rqa_xy.empty: combined_groups.update(df_rqa_xy['Zkusenosti'].dropna().unique())
    all_categories = sorted(list(combined_groups))
    extra_palette = sns.color_palette("Set2", max(len(all_categories), 1)).as_hex()
    for i, cat in enumerate(all_categories):
        if cat not in EXPERIENCE_COLORS:
            EXPERIENCE_COLORS[cat] = extra_palette[i % len(extra_palette)]

    # --- UI STRUKTURA ---
    out_rp = widgets.Output()
    out_box = widgets.Output()
    out_density = widgets.Output()
    out_radar = widgets.Output()

    tab = widgets.Tab(children=[out_rp, out_box, out_density, out_radar])
    tab.set_title(0, 'Recurrence plot')
    tab.set_title(1, 'Boxplot')
    tab.set_title(2, 'Scatter plot')
    tab.set_title(3, 'Radar plot')

    display(dd_rqa_mode, tab)

    # Inicializace seznamů pro UI
    active_initial = get_active_data()
    available_stimuli = ['All'] + sorted(active_initial['stimulus'].dropna().unique().tolist())
    available_respondents = ['Všichni'] + sorted(active_initial['participant'].dropna().unique().tolist())

    # ==========================================
    # TAB 1: RECURRENCE PLOT
    # ==========================================
    with out_rp:
        AOI_COLORS = {
            'Brightest': '#7c68f4', 'Darkest': '#4a3bc9', 'Target': '#1f991f',
            'Others': '#5e5e66', 'Title': '#4b4c53', 'Imprint': '#2e2f37',
            '1': '#fff5f0', '2': '#fee0d2', '3': '#fcbba1',
            '4': '#fc9272', '5': '#fb6a4a', '6': '#ef3b2c',
            '7': '#cb181d', '8': '#a50f15', '9': '#67000d'
        }

        plot_data_rp = df_complete.copy()
        plot_data_rp['AOI Single'] = plot_data_rp['AOI Single'].astype(str)

        # Předpřipravení palety pro AOI (univerzální)
        all_aois = sorted(plot_data_rp['AOI Single'].unique().tolist())
        color_palette = sns.color_palette("husl", len(all_aois)).as_hex()
        colors_map = dict(zip(all_aois, color_palette))
        for aoi_name, aoi_color in AOI_COLORS.items():
            if aoi_name in colors_map: colors_map[aoi_name] = aoi_color

        respondents_rp = sorted(plot_data_rp['participant'].dropna().unique())
        stimuli_rp = sorted(plot_data_rp['stimulus'].dropna().unique())

        dd_resp_rp = widgets.Dropdown(options=respondents_rp, description='Respondent:')
        dd_stim_rp = widgets.Dropdown(options=stimuli_rp, description='Stimul:')
        btn_rp = widgets.Button(description='Vykreslit RP', button_style='primary')
        out_graph_rp = widgets.Output()

        def plot_rp(b):
            with out_graph_rp:
                clear_output(wait=True)
                subset = plot_data_rp[(plot_data_rp['participant'] == dd_resp_rp.value) &
                                      (plot_data_rp['stimulus'] == dd_stim_rp.value)].sort_values('timestamp').copy()

                if subset.empty:
                    print("Žádná data pro tuto kombinaci.")
                    return

                is_xy_mode = 'XY' in dd_rqa_mode.value
                fig, ax = plt.subplots(figsize=(8, 7))
                sequence_len = len(subset)

                # Podbarvení Target Intervalu (odolné vůči None a chybějícím cílům)
                target_interval = None
                if 'Target_Interval' in subset.columns and pd.notna(subset['Target_Interval'].iloc[0]) and str(subset['Target_Interval'].iloc[0]).strip() != '':
                    target_interval = str(subset['Target_Interval'].iloc[0])
                    seq_target = np.array(subset['AOI Single'].tolist())
                    target_indices = np.where(seq_target == target_interval)[0]

                    for idx in target_indices:
                        ax.axvspan(idx + 0.5, idx + 1.5, color='lightgreen', alpha=0.25, lw=0)
                        ax.axhspan(idx + 0.5, idx + 1.5, color='lightgreen', alpha=0.25, lw=0)

                # Vykreslení bodů
                if is_xy_mode:
                    coords = np.column_stack((pd.to_numeric(subset['x'], errors='coerce'), pd.to_numeric(subset['y'], errors='coerce')))
                    active_rqa = get_active_data()
                    radius = active_rqa['Radius_px'].iloc[0] if 'Radius_px' in active_rqa.columns else 50
                    dist_matrix = cdist(coords, coords, metric='euclidean')
                    rows, cols = np.where(dist_matrix <= radius)
                    ax.scatter(cols + 1, rows + 1, color='#1f77b4', s=25, alpha=0.9, edgecolors='none', zorder=3)
                else:
                    sequence = subset['AOI Single'].tolist()
                    seq = np.array(sequence)
                    rows, cols = np.where(seq[:, None] == seq[None, :])
                    point_colors = [colors_map.get(seq[r], '#000000') for r in rows]
                    ax.scatter(cols + 1, rows + 1, c=point_colors, s=25, alpha=0.9, edgecolors='none', zorder=3)

                # Titulek a osy
                exp = subset['Zkusenosti'].iloc[0] if 'Zkusenosti' in subset.columns else 'Neznámá'
                title_text = f"Recurrence Plot: {dd_resp_rp.value} ({exp})\nStimul: {dd_stim_rp.value} | Celkem fixací: {sequence_len}"
                if target_interval: title_text += f"\nDotazovaný interval: {target_interval}"

                ax.set_title(title_text, fontsize=13, fontweight='bold', pad=15)
                ax.set_aspect('equal')
                ax.set_xlim(0, sequence_len + 1); ax.set_ylim(0, sequence_len + 1)

                step = 2 if sequence_len <= 25 else 5
                ax.xaxis.set_major_locator(ticker.MultipleLocator(step))
                ax.yaxis.set_major_locator(ticker.MultipleLocator(step))
                ax.grid(True, linestyle=':', alpha=0.4, zorder=0)
                ax.set_xlabel('Pořadí fixace (i)', fontsize=11)
                ax.set_ylabel('Pořadí fixace (j)', fontsize=11)

                # Kategorizovaná legenda s fallbackem pro neznámé AOI
                if not is_xy_mode:
                    present_aois = sorted(list(set(sequence)))
                    map_field_labels = {'Brightest': 'Nejsvětlejší oblast', 'Darkest': 'Nejtmavší oblast', 'Target': 'Dotazovaná oblast', 'Others': 'Zbylý obsah'}
                    other_elements_labels = {'Title': 'Titul', 'Imprint': 'Tiráž'}

                    map_field_aois = [k for k in present_aois if k in map_field_labels]
                    other_elements_aois = [k for k in present_aois if k in other_elements_labels]
                    interval_aois = [k for k in present_aois if any(c.isdigit() for c in k)]

                    if 'Others' in map_field_aois:
                        map_field_aois.remove('Others'); map_field_aois.append('Others')

                    legend_elements = []

                    if map_field_aois:
                        legend_elements.append(Line2D([0], [0], color='none', label=r'$\bf{Mapové\ pole}$'))
                        for k in map_field_aois:
                            legend_elements.append(Line2D([0], [0], marker='o', color='w', label=f"  {map_field_labels[k]}", markerfacecolor=colors_map[k], markersize=10))

                    if interval_aois or target_interval:
                        legend_elements.append(Line2D([0], [0], color='none', label=r'$\bf{Legenda}$'))
                        for k in interval_aois:
                            legend_elements.append(Line2D([0], [0], marker='o', color='w', label=f"  {k}", markerfacecolor=colors_map[k], markersize=10))
                        if target_interval:
                            legend_elements.append(Line2D([0], [0], marker='s', color='w', label="  Dotazovaný interval", markerfacecolor='lightgreen', alpha=0.5, markersize=12))

                    if other_elements_aois:
                        legend_elements.append(Line2D([0], [0], color='none', label=r'$\bf{Ostatní\ prvky}$'))
                        for k in other_elements_aois:
                            legend_elements.append(Line2D([0], [0], marker='o', color='w', label=f"  {other_elements_labels[k]}", markerfacecolor=colors_map[k], markersize=10))

                    # Vykreslení neznámých AOI do legendy
                    unknown_aois = [k for k in present_aois if k not in map_field_labels and k not in other_elements_labels and not any(c.isdigit() for c in k)]
                    if unknown_aois:
                        legend_elements.append(Line2D([0], [0], color='none', label=r'$\bf{Vlastní\ oblasti}$'))
                        for k in unknown_aois:
                            legend_elements.append(Line2D([0], [0], marker='o', color='w', label=f"  {k}", markerfacecolor=colors_map[k], markersize=10))

                    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1))

                plt.tight_layout()
                plt.show()

        btn_rp.on_click(plot_rp)
        display(widgets.HBox([dd_resp_rp, dd_stim_rp, btn_rp]), out_graph_rp)

    # ==========================================
    # TAB 2: BOXPLOTY
    # ==========================================
    with out_box:
        dd_stim_box = widgets.Dropdown(options=available_stimuli, description='Stimul:')
        btn_box = widgets.Button(description='Vykreslit Boxploty', button_style='primary')
        out_graph_box = widgets.Output()

        def plot_box(b):
            with out_graph_box:
                clear_output(wait=True)
                df = get_active_data()
                if dd_stim_box.value != 'All': df = df[df['stimulus'] == dd_stim_box.value]

                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                groups = sorted(df['Zkusenosti'].dropna().unique().tolist())

                for i, m in enumerate(['RR', 'DET', 'LAM']):
                    sns.boxplot(data=df, x='Zkusenosti', y=m, ax=axes[i], palette=EXPERIENCE_COLORS, order=groups, width=0.5)
                    axes[i].set_ylim(-0.05, 1.05)
                    axes[i].set_title(m, fontweight='bold')
                    axes[i].set_xlabel('')
                    axes[i].grid(True, axis='y', linestyle=':', alpha=0.6)

                    medians = df.groupby('Zkusenosti')[m].median()
                    for tick, label in enumerate(groups):
                        if pd.notna(medians.get(label)):
                            med_val = medians[label]
                            txt = axes[i].text(tick, med_val, f'{med_val:.3f}',
                                               ha='center', va='center', fontweight='bold', color='black', fontsize=11)
                            txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

                fig.suptitle("Porovnání metrik", fontweight='bold', fontsize=14)

                plt.tight_layout()
                plt.show()

        btn_box.on_click(plot_box)
        display(widgets.HBox([dd_stim_box, btn_box]), out_graph_box)

    # ==========================================
    # TAB 3: Scatter plot
    # ==========================================
    with out_density:
        dd_resp_dens = widgets.Dropdown(options=available_respondents, description='Respondent:')
        dd_stim_dens = widgets.Dropdown(options=available_stimuli, description='Stimul:')
        dd_agg_dens = widgets.Dropdown(options=[('Jednotlivá měření', 'raw'), ('Průměry za respondenta', 'agg')], description='Režim:')
        btn_dens = widgets.Button(description='Vykreslit strategii', button_style='primary')
        out_graph_dens = widgets.Output()

        def plot_dens(b):
            with out_graph_dens:
                clear_output(wait=True)
                df = get_active_data()

                if dd_agg_dens.value == 'raw' and dd_stim_dens.value != 'All':
                    df = df[df['stimulus'] == dd_stim_dens.value]
                if dd_resp_dens.value != 'Všichni':
                    df = df[df['participant'] == dd_resp_dens.value]

                if dd_agg_dens.value == 'agg':
                    df = df.groupby(['participant', 'Zkusenosti'])[['RR', 'DET', 'LAM']].mean().reset_index()

                if df.empty:
                    print("Žádná data pro tuto volbu.")
                    return

                fig, ax = plt.subplots(figsize=(9, 6))
                sns.scatterplot(data=df, x='DET', y='LAM', hue='Zkusenosti', size='RR',
                                sizes=(50, 500), palette=EXPERIENCE_COLORS, alpha=0.6, ax=ax, edgecolor='white')
                ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
                ax.set_title("Vizuální strategie (DET vs LAM)", fontweight='bold')
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                ax.grid(True, linestyle=':', alpha=0.6)
                plt.show()

        btn_dens.on_click(plot_dens)
        display(widgets.VBox([widgets.HBox([dd_resp_dens, dd_stim_dens, dd_agg_dens]), btn_dens, out_graph_dens]))

    # ==========================================
    # TAB 4: RADAR plot
    # ==========================================
    with out_radar:
        dd_resp_rad = widgets.Dropdown(options=available_respondents, description='Respondent:')
        dd_stim_rad = widgets.Dropdown(options=available_stimuli, description='Stimul:')
        btn_rad = widgets.Button(description='Vykreslit Radar', button_style='primary')
        out_graph_rad = widgets.Output()

        def plot_radar(b):
            with out_graph_rad:
                clear_output(wait=True)
                df = get_active_data()

                if dd_stim_rad.value != 'All':
                    df = df[df['stimulus'] == dd_stim_rad.value]
                if dd_resp_rad.value != 'Všichni':
                    df = df[df['participant'] == dd_resp_rad.value]

                if df.empty:
                    print("Pro tuto volbu nejsou k dispozici žádná data.")
                    return

                avg = df.groupby('Zkusenosti')[['RR', 'DET', 'LAM']].mean()

                categories = ['RR', 'DET', 'LAM']
                N = len(categories)
                angles = [n / float(N) * 2 * np.pi for n in range(N)]
                angles += angles[:1]

                fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
                for idx, row in avg.iterrows():
                    values = row.values.flatten().tolist()
                    values += values[:1]
                    ax.plot(angles, values, label=idx, color=EXPERIENCE_COLORS.get(idx, '#333333'), linewidth=2)
                    ax.fill(angles, values, alpha=0.15, color=EXPERIENCE_COLORS.get(idx, '#333333'))

                ax.set_xticks(angles[:-1])
                ax.set_xticklabels(categories, size=12, fontweight='bold')
                ax.set_ylim(0, 1)

                plot_title = "Všechny stimuly" if dd_stim_rad.value == 'All' else f"Stimul: {dd_stim_rad.value}"
                if dd_resp_rad.value != 'Všichni':
                    plot_title += f" | Resp: {dd_resp_rad.value}"

                plt.title(f'Profil RQA strategií\n({plot_title})', size=14, fontweight='bold', y=1.1)
                plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), title="Zkušenost")
                plt.tight_layout()
                plt.show()

        btn_rad.on_click(plot_radar)
        display(widgets.HBox([dd_resp_rad, dd_stim_rad, btn_rad]), out_graph_rad)

In [ ]:
# @title Vliv složitosti legendy (počet intervalů) na RQA metriky { display-mode: "form" }
# @markdown Tento vizualizační blok se zaměřuje na vliv vizuální a informační náročnosti stimulů. Mapy jsou automaticky seskupeny podle počtu tříd (intervalů) v legendě, čímž vzniká proměnná vyjadřující „složitost úkolu“.
# @markdown * **Hlavní panel (Boxploty):** Zobrazuje plošné srovnání metrik RQA pro všechny dostupné úrovně složitosti. Vlevo jsou výsledky sémantického přístupu (AOI), vpravo výsledky prostorového měření (XY). Sledujte zejména trendy u mediánových hodnot – jak se mění Laminarita (zasekávání zraku) či Determinismus (systémovost), když do legendy přidáme více barevných tříd.
# @markdown * **Spodní panel (Scatter ploty):** Interaktivní rozhraní pro zkoumání individuálních rozdílů. Zatímco horní grafy agregují data za celou skupinu, tento nástroj umožňuje vykreslit bodový graf strategií (DET vs. LAM) pro konkrétního respondenta nebo vybranou skupinu s barevným odlišením podle toho, s jak složitým stimulem zrovna pracovali.
# --- 1. NAČTĚNÍ DAT ---
try:
    folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    file_rqa_aoi = FILE_OUT_AOI if 'FILE_OUT_AOI' in globals() else folder_path + 'rqa_results_AOI.csv'
    file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
except NameError:
    pass

df_rqa_aoi = pd.read_csv(file_rqa_aoi) if os.path.exists(file_rqa_aoi) else pd.DataFrame()
df_rqa_xy = pd.read_csv(file_rqa_xy) if os.path.exists(file_rqa_xy) else pd.DataFrame()

if df_rqa_aoi.empty and df_rqa_xy.empty:
    print("Žádná vypočítaná RQA data nebyla nalezena. Spusťte prosím KROK 7.")
    raise ValueError("Chybějící RQA data")

# Funkce pro přípravu a vyčištění dat (extrakce intervalů)
def prepare_plot_data(df):
    if df.empty:
        return df
    df_clean = df.dropna(subset=['RR', 'DET', 'LAM']).copy()
    df_clean['Num_Intervals'] = df_clean['stimulus'].str.extract(r'R(\d+)')[0]
    df_clean['Num_Intervals'] = pd.to_numeric(df_clean['Num_Intervals'], errors='coerce')
    df_clean = df_clean.dropna(subset=['Num_Intervals'])
    if not df_clean.empty:
        df_clean['Num_Intervals'] = df_clean['Num_Intervals'].astype(int)
    return df_clean

df_plot_aoi = prepare_plot_data(df_rqa_aoi)
df_plot_xy = prepare_plot_data(df_rqa_xy)

# Sjednocení skupin pro konzistentní barvy
experience_col = 'Zkusenosti'
all_groups = set()
if not df_plot_aoi.empty: all_groups.update(df_plot_aoi[experience_col].unique())
if not df_plot_xy.empty: all_groups.update(df_plot_xy[experience_col].unique())
groups = sorted(list(all_groups))

if 'EXPERIENCE_COLORS' not in globals():
    EXPERIENCE_COLORS = {'Nováček': '#ff7f0e', 'Expert': '#1f77b4'}

extra_palette = sns.color_palette("Set2", max(len(groups), 1)).as_hex()
for i, cat in enumerate(groups):
    if cat not in EXPERIENCE_COLORS:
        EXPERIENCE_COLORS[cat] = extra_palette[i % len(extra_palette)]

# --- 2. VYKRESLENÍ BOXPLOTŮ (VEDLE SEBE) ---
out_boxplot = widgets.Output()
with out_boxplot:
    # 3 řádky (RR, DET, LAM) x 2 sloupce (AOI, XY)
    fig, axes = plt.subplots(3, 2, figsize=(20, 18))
    fig.suptitle("Vliv složitosti legendy (počtu intervalů) na RQA metriky\nSrovnání AOI přístupu a XY přístupu", fontsize=18, fontweight='bold', y=1.02)

    metrics = ['RR', 'DET', 'LAM']
    axis_titles = ['Recurrence Rate (RR)', 'Determinismus (DET)', 'Laminarita (LAM)']
    n_hues = len(groups)

    for i, metric in enumerate(metrics):
        # Levý sloupec: AOI
        if not df_plot_aoi.empty:
            intervals_aoi = sorted(df_plot_aoi['Num_Intervals'].unique().tolist())
            sns.boxplot(
                data=df_plot_aoi, x='Num_Intervals', y=metric, hue=experience_col, ax=axes[i, 0],
                palette=EXPERIENCE_COLORS, order=intervals_aoi, hue_order=groups,
                width=0.6, boxprops=dict(alpha=0.85), flierprops=dict(marker='o', markersize=4, alpha=0.5)
            )
            axes[i, 0].set_title(f"AOI přístup - {axis_titles[i]}", fontsize=14, fontweight='bold', pad=10)
            axes[i, 0].set_xlabel("Počet intervalů v legendě", fontsize=12)
            axes[i, 0].set_ylabel("Hodnota metriky", fontsize=12)

            # Zajištění jednotného měřítka Y osy (0 až 1)
            axes[i, 0].set_ylim(-0.05, 1.05)

            axes[i, 0].grid(True, linestyle=':', alpha=0.6, axis='y')
            axes[i, 0].get_legend().remove()

            # Mediány AOI
            for x_idx, interval_val in enumerate(intervals_aoi):
                for hue_idx, group in enumerate(groups):
                    offset = (hue_idx - (n_hues - 1) / 2) * (0.6 / n_hues)
                    subset = df_plot_aoi[(df_plot_aoi['Num_Intervals'] == interval_val) & (df_plot_aoi[experience_col] == group)]
                    if not subset.empty:
                        med_val = subset[metric].median()
                        txt = axes[i, 0].text(
                            x_idx + offset, med_val, f'{med_val:.3f}',
                            horizontalalignment='center', verticalalignment='center',
                            fontsize=10, fontweight='bold', color='black', zorder=10
                        )
                        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white', alpha=0.9)])

        # Pravý sloupec: XY
        if not df_plot_xy.empty:
            intervals_xy = sorted(df_plot_xy['Num_Intervals'].unique().tolist())
            sns.boxplot(
                data=df_plot_xy, x='Num_Intervals', y=metric, hue=experience_col, ax=axes[i, 1],
                palette=EXPERIENCE_COLORS, order=intervals_xy, hue_order=groups,
                width=0.6, boxprops=dict(alpha=0.85), flierprops=dict(marker='o', markersize=4, alpha=0.5)
            )
            axes[i, 1].set_title(f"XY přístup - {axis_titles[i]}", fontsize=14, fontweight='bold', pad=10)
            axes[i, 1].set_xlabel("Počet intervalů v legendě", fontsize=12)
            axes[i, 1].set_ylabel("Hodnota metriky", fontsize=12)

            # Zajištění jednotného měřítka Y osy (0 až 1)
            axes[i, 1].set_ylim(-0.05, 1.05)

            axes[i, 1].grid(True, linestyle=':', alpha=0.6, axis='y')

            if i == 2:
                axes[i, 1].legend(title="Zkušenost", loc='upper right', bbox_to_anchor=(1.25, 1), fontsize=11)
            else:
                axes[i, 1].get_legend().remove()

            # Mediány XY
            for x_idx, interval_val in enumerate(intervals_xy):
                for hue_idx, group in enumerate(groups):
                    offset = (hue_idx - (n_hues - 1) / 2) * (0.6 / n_hues)
                    subset = df_plot_xy[(df_plot_xy['Num_Intervals'] == interval_val) & (df_plot_xy[experience_col] == group)]
                    if not subset.empty:
                        med_val = subset[metric].median()
                        txt = axes[i, 1].text(
                            x_idx + offset, med_val, f'{med_val:.3f}',
                            horizontalalignment='center', verticalalignment='center',
                            fontsize=10, fontweight='bold', color='black', zorder=10
                        )
                        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white', alpha=0.9)])

    plt.tight_layout()
    plt.show()

# --- 3. INTERAKTIVNÍ SCATTER PLOT (VEDLE SEBE) ---
out_scatter = widgets.Output()
with out_scatter:
    print("-" * 100)
    lbl_scatter = widgets.Label("Individuální strategie: DET a LAM podle složitosti", layout=widgets.Layout(font_weight='bold'))

    group_choices = [f"Pouze {g}" for g in groups]
    individuals = set()
    if not df_plot_aoi.empty: individuals.update(df_plot_aoi['participant'].dropna().unique())
    if not df_plot_xy.empty: individuals.update(df_plot_xy['participant'].dropna().unique())

    available_options = ['Všichni'] + group_choices + sorted(list(individuals))

    dd_selection = widgets.Dropdown(options=available_options, description='Výběr dat:', style={'description_width': 'initial'})
    btn_scatter = widgets.Button(description='Vykreslit Scatter Plot', button_style='info', layout=widgets.Layout(width='200px'))
    out_graph_scatter = widgets.Output()

    def plot_scatter(b):
        with out_graph_scatter:
            clear_output(wait=True)
            selection = dd_selection.value

            # Filtrace dat
            if selection == 'Všichni':
                data_scat_aoi = df_plot_aoi.copy()
                data_scat_xy = df_plot_xy.copy()
                title_suffix = "Všichni respondenti"
            elif selection.startswith('Pouze '):
                group_name = selection.replace('Pouze ', '')
                data_scat_aoi = df_plot_aoi[df_plot_aoi[experience_col] == group_name].copy() if not df_plot_aoi.empty else pd.DataFrame()
                data_scat_xy = df_plot_xy[df_plot_xy[experience_col] == group_name].copy() if not df_plot_xy.empty else pd.DataFrame()
                title_suffix = f"Skupina: {group_name}"
            else:
                data_scat_aoi = df_plot_aoi[df_plot_aoi['participant'] == selection].copy() if not df_plot_aoi.empty else pd.DataFrame()
                data_scat_xy = df_plot_xy[df_plot_xy['participant'] == selection].copy() if not df_plot_xy.empty else pd.DataFrame()
                title_suffix = f"Respondent: {selection}"

            if data_scat_aoi.empty and data_scat_xy.empty:
                print("Nejsou k dispozici žádná data pro tento výběr.")
                return

            fig, axes = plt.subplots(1, 2, figsize=(18, 7))
            fig.suptitle(f"Rozložení strategií v závislosti na složitosti legendy ({title_suffix})", fontsize=16, fontweight='bold')

            # Levý graf: AOI
            if not data_scat_aoi.empty:
                sns.scatterplot(
                    data=data_scat_aoi, x='DET', y='LAM', hue='Num_Intervals',
                    palette='viridis', s=120, alpha=0.85, edgecolor='black', ax=axes[0]
                )
                axes[0].set_title("AOI přístup", fontsize=14, fontweight='bold', pad=10)
                axes[0].set_xlabel("Determinismus (DET)", fontsize=12, fontweight='bold')
                axes[0].set_ylabel("Laminarita (LAM)", fontsize=12, fontweight='bold')

                axes[0].set_xlim(-0.05, 1.05)
                axes[0].set_ylim(-0.05, 1.05)
                axes[0].grid(True, linestyle=':', alpha=0.6)
                axes[0].get_legend().remove()
            else:
                axes[0].text(0.5, 0.5, 'Žádná data pro AOI', ha='center', va='center', fontsize=12)

            # Pravý graf: XY
            if not data_scat_xy.empty:
                sns.scatterplot(
                    data=data_scat_xy, x='DET', y='LAM', hue='Num_Intervals',
                    palette='viridis', s=120, alpha=0.85, edgecolor='black', ax=axes[1]
                )
                axes[1].set_title("XY přístup", fontsize=14, fontweight='bold', pad=10)
                axes[1].set_xlabel("Determinismus (DET)", fontsize=12, fontweight='bold')
                axes[1].set_ylabel("Laminarita (LAM)", fontsize=12, fontweight='bold')

                axes[1].set_xlim(-0.05, 1.05)
                axes[1].set_ylim(-0.05, 1.05)
                axes[1].grid(True, linestyle=':', alpha=0.6)

                handles, labels = axes[1].get_legend_handles_labels()
                axes[1].legend(handles=handles, labels=[f"{l} intervalů" for l in labels], title="Složitost mapy", bbox_to_anchor=(1.05, 1), loc='upper left')
            else:
                axes[1].text(0.5, 0.5, 'Žádná data pro XY', ha='center', va='center', fontsize=12)

            plt.tight_layout()
            plt.show()

    btn_scatter.on_click(plot_scatter)

display(out_boxplot)
display(widgets.VBox([lbl_scatter, widgets.HBox([dd_selection, btn_scatter]), out_graph_scatter]))

In [ ]:
# @title Porovnání skupin se shluky vytvořené dle K-Means automatického shlukování { display-mode: "form" }
# @markdown Tento krok využívá algoritmus K-Means k automatické detekci skrytých kognitivních vzorců a vizuálních strategií v datech, a to zcela nezávisle na předem definovaném rozdělení respondentů na experty a nováčky.
# @markdown * **Úroveň analýzy:** Zvolte, zda chcete do shluků rozřadit jednotlivé pokusy (jak probíhalo čtení každé jednotlivé mapy nezávisle na sobě), nebo celkové agregované profily (průměrné chování a strategie každého respondenta napříč celým experimentem).
# @markdown * **Počet shluků (k):** Posuvníkem definujte, kolik odlišných kognitivních profilů má algoritmus v datech hledat.
# @markdown * **Princip výpočtu:** Skript z obou přístupů (AOI i XY) extrahuje vypočtené RQA metriky (RR, DET, LAM). Data nejprve matematicky standardizuje a následně je iterativně rozdělí do zadaného počtu skupin tak, aby si byly vizuální strategie uvnitř jednoho shluku co nejvíce podobné.
# @markdown * **Výstup:** Po úspěšném výpočtu se pod buňkou vygenerují tabulky s průměrnými hodnotami metrik pro jednotlivé shluky. Tato segmentovaná data se automaticky uloží do paměti Colabu a poslouží jako vstup pro detailní grafickou interpretaci v navazujícím kroku.
# --- UŽIVATELSKÉ ROZHRANÍ ---
lbl_cluster = widgets.Label("Nastavení výpočtu globálního modelu K-Means pro oba přístupy (AOI i XY):", layout=widgets.Layout(font_weight='bold', font_size='14px'))
num_clusters = widgets.IntSlider(value=2, min=2, max=6, description='Počet shluků (k):', style={'description_width': 'initial'})
dd_agg = widgets.Dropdown(
    options=['Jednotlivá měření (1 tečka = 1 pokus)', 'Celkový profil respondenta (1 tečka = 1 člověk)'],
    description='Úroveň analýzy:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
btn_cluster = widgets.Button(description='Spočítat shluky', button_style='primary', layout=widgets.Layout(width='250px', margin='10px 0 0 0'))
out_cluster = widgets.Output()

# --- FUNKCE PRO ZPRACOVÁNÍ JEDNOHO DATASETU ---
def process_clustering(data_rqa, is_aggregated, k_clusters):
    data_clean = data_rqa.dropna(subset=['RR', 'DET', 'LAM']).copy()

    if is_aggregated:
        group_cols = ['participant']
        if 'Zkusenosti' in data_clean.columns:
            group_cols.append('Zkusenosti')

        # Zprůměrování metrik pro každého respondenta
        data_clean = data_clean.groupby(group_cols)[['RR', 'DET', 'LAM']].mean().reset_index()
        data_clean['stimulus'] = 'Všechny (Agregováno)'
        data_clean['Num_Intervals'] = np.nan
    else:
        # Extrakce složitosti (pokud není agregováno)
        data_clean['Num_Intervals'] = data_clean['stimulus'].str.extract(r'R(\d+)')[0]
        data_clean['Num_Intervals'] = pd.to_numeric(data_clean['Num_Intervals'], errors='coerce')

    if len(data_clean) < k_clusters:
        return None, None, f"Nedostatek dat pro {k_clusters} shluky. V datasetu je pouze {len(data_clean)} záznamů."

    # Standardizace a K-Means
    features = ['RR', 'DET', 'LAM']
    X = data_clean[features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
    data_clean['Cluster'] = ["Shluk " + str(c + 1) for c in kmeans.fit_predict(X_scaled)]

    cluster_means = data_clean.groupby('Cluster')[['RR', 'DET', 'LAM']].mean().round(3)

    meta_info = {
        'stim': 'Agregovaný profil' if is_aggregated else 'Globální model (jednotlivá měření)',
        'k': k_clusters,
        'aggregated': is_aggregated
    }

    return data_clean, meta_info, cluster_means


# --- HLAVNÍ BĚH ---
def on_cluster_clicked(b):
    with out_cluster:
        clear_output(wait=True)
        print("Načítám kompletní RQA data...")

        try:
            folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
            file_rqa_aoi = FILE_OUT_AOI if 'FILE_OUT_AOI' in globals() else folder_path + 'rqa_results_AOI.csv'
            file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
        except NameError:
            folder_path = "drive/MyDrive/BAK-RQA/"
            file_rqa_aoi = folder_path + 'rqa_results_AOI.csv'
            file_rqa_xy = folder_path + 'rqa_results_XY.csv'

        df_rqa_aoi = pd.read_csv(file_rqa_aoi) if os.path.exists(file_rqa_aoi) else pd.DataFrame()
        df_rqa_xy = pd.read_csv(file_rqa_xy) if os.path.exists(file_rqa_xy) else pd.DataFrame()

        if df_rqa_aoi.empty and df_rqa_xy.empty:
            print("Chyba: Žádná RQA data nebyla nalezena. Spusťte nejprve výpočet v KROKU 7.")
            return

        is_aggregated = 'Celkový profil' in dd_agg.value
        k_clusters = num_clusters.value

        print(f"Úroveň analýzy: {'Agregováno za respondenta' if is_aggregated else 'Jednotlivá měření'}")
        print(f"Počet hledaných shluků: {k_clusters}")
        print("-" * 60)

        # 1. Zpracování AOI přístupu
        if not df_rqa_aoi.empty:
            print("Příprava a výpočet K-Means pro AOI přístup...")
            df_clustered_aoi, meta_aoi, means_aoi = process_clustering(df_rqa_aoi, is_aggregated, k_clusters)

            if df_clustered_aoi is not None:
                globals()['df_clustered_aoi'] = df_clustered_aoi
                globals()['cluster_meta_aoi'] = meta_aoi
                print("Průměrné hodnoty RQA metrik pro jednotlivé shluky (AOI):")
                display(means_aoi)
            else:
                print(means_aoi)
        else:
            print("Data pro AOI přístup nejsou k dispozici.")

        print("-" * 60)

        # 2. Zpracování XY přístupu
        if not df_rqa_xy.empty:
            print("Příprava a výpočet K-Means pro XY přístup...")
            df_clustered_xy, meta_xy, means_xy = process_clustering(df_rqa_xy, is_aggregated, k_clusters)

            if df_clustered_xy is not None:
                globals()['df_clustered_xy'] = df_clustered_xy
                globals()['cluster_meta_xy'] = meta_xy
                print("Průměrné hodnoty RQA metrik pro jednotlivé shluky (XY):")
                display(means_xy)
            else:
                print(means_xy)
        else:
            print("Data pro XY přístup nejsou k dispozici.")

        print("-" * 60)
        print("Výpočet úspěšně dokončen! Globální kognitivní vzorce byly nalezeny pro oba přístupy.")
        print("Nyní můžete přejít na KROK 9b a tyto shluky graficky vizualizovat.")

btn_cluster.on_click(on_cluster_clicked)
display(widgets.VBox([lbl_cluster, dd_agg, num_clusters, btn_cluster, out_cluster]))

In [ ]:
# @title Vizualizace shluků { display-mode: "form" }
# @markdown Tento blok kódu zajišťuje grafickou vizualizaci vypočtených shluků a jejich porovnání s reálnými parametry experimentu. Prostřednictvím tří vedle sebe umístěných scatter plotů můžete zkoumat, zda matematicky detekované kognitivní vzorce (shluky) reálně odpovídají zkušenostem uživatelů nebo složitosti testovaných map.
# @markdown * **Přepínač a filtry:** Můžete plynule přepínat mezi vizualizací AOI a XY přístupu. Filtry umožňují grafy omezit jen na specifické respondenty, stimuly, nebo úroveň zkušeností. (Pozn.: filtry pouze mění rozsah zobrazených bodů, ale nemění podstatu shluků vypočítanou v předchozím kroku).
# @markdown * **Graf 1 (Zkušenosti):** Zobrazuje rozložení strategií s barevným kódováním podle reálné zkušenosti respondentů (experti vs. nováčci).
# @markdown * **Graf 2 (Shluky):** Totéž rozložení bodů, ale barevně odlišené podle příslušnosti ke shlukům, které zcela autonomně nalezl algoritmus strojového učení (K-Means). Porovnáním prvního a druhého grafu zjistíte, zda algoritmus odhalil "odbornost" bez znalosti kontextu.
# @markdown * **Graf 3 (Složitost):** Pokud je vizualizace provedena na úrovni jednotlivých měření, třetí graf odhalí, jakou roli hraje složitost mapy (počet intervalů legendy) v posunech mezi těmito strategiemi.
# @markdown * **Měřítko:** Velikost bublin napříč všemi grafy i pohledy je ukotvena v absolutním globálním minimu a maximu RR, což garantuje vizuální konzistenci při aplikaci jakýchkoliv filtrů.
has_aoi = 'df_clustered_aoi' in globals() and 'cluster_meta_aoi' in globals()
has_xy = 'df_clustered_xy' in globals() and 'cluster_meta_xy' in globals()

if not has_aoi and not has_xy:
    print("Pozor: V paměti nejsou žádná data ke shlukování.")
    print("Prosím, jděte nejprve na KROK 9a, nastavte parametry a klikněte na tlačítko pro výpočet shluků.")
else:
    options_mode = []
    if has_aoi: options_mode.append('AOI přístup')
    if has_xy: options_mode.append('XY přístup')

    # --- PŘEPÍNAČ REŽIMU ---
    dd_rqa_mode = widgets.Dropdown(options=options_mode, description='Režim RQA:', style={'description_width': 'initial'})

    def get_active_cluster_data():
        if 'AOI' in dd_rqa_mode.value:
            return globals()['df_clustered_aoi'].copy(), globals()['cluster_meta_aoi']
        else:
            return globals()['df_clustered_xy'].copy(), globals()['cluster_meta_xy']

    # --- PŘÍPRAVA MOŽNOSTÍ PRO FILTRY A GLOBÁLNÍHO MĚŘÍTKA ---
    all_participants = set()
    all_stimuli = set()
    all_experience = set()
    all_intervals = set()
    rr_all_values = []

    for df_temp in [globals().get('df_clustered_aoi', pd.DataFrame()), globals().get('df_clustered_xy', pd.DataFrame())]:
        if not df_temp.empty:
            all_participants.update(df_temp['participant'].dropna().unique())
            all_stimuli.update(df_temp['stimulus'].dropna().unique())
            if 'Zkusenosti' in df_temp.columns:
                all_experience.update(df_temp['Zkusenosti'].dropna().unique())
            if 'Num_Intervals' in df_temp.columns:
                all_intervals.update(df_temp['Num_Intervals'].dropna().unique())
            if 'RR' in df_temp.columns:
                rr_all_values.extend(df_temp['RR'].dropna().tolist())

    # Získání globálního minima a maxima RR pro jednotné měřítko bublin
    global_rr_min = min(rr_all_values) if rr_all_values else 0.0
    global_rr_max = max(rr_all_values) if rr_all_values else 1.0

    available_resp = ['Všichni'] + sorted(list(all_participants))
    available_stim = ['Všechny'] + sorted(list(all_stimuli))
    available_exp = ['Všechny'] + sorted(list(all_experience)) if all_experience else ['Všechny']
    available_int = ['Všechny'] + [f"{int(i)} intervalů" for i in sorted(list(all_intervals))] if all_intervals else ['Všechny']

    # --- UI PRVKY PRO FILTROVÁNÍ ---
    lbl_filters = widgets.Label("Filtrování vypočtených dat (nemění samotné shluky):", layout=widgets.Layout(font_weight='bold', font_size='14px'))
    dd_resp_vis = widgets.Dropdown(options=available_resp, description='Respondent:', style={'description_width': 'initial'})
    dd_exp_vis = widgets.Dropdown(options=available_exp, description='Zkušenost:', style={'description_width': 'initial'})

    dd_stim_vis = widgets.Dropdown(options=available_stim, description='Stimul:', style={'description_width': 'initial'})
    dd_int_vis = widgets.Dropdown(options=available_int, description='Složitost:', style={'description_width': 'initial'})

    btn_plot = widgets.Button(description='Vykreslit grafy', button_style='info', layout=widgets.Layout(width='200px', margin='10px 0 0 0'))
    out_plot = widgets.Output()

    # Dynamické vypnutí filtrů podle úrovně agregace zvoleného modelu
    def update_ui_state(*args):
        _, meta = get_active_cluster_data()
        is_aggregated = meta.get('aggregated', False)
        dd_stim_vis.disabled = is_aggregated
        dd_int_vis.disabled = is_aggregated

    dd_rqa_mode.observe(update_ui_state, 'value')
    update_ui_state()

    def on_plot_clicked(b):
        with out_plot:
            clear_output(wait=True)
            df_plot, meta = get_active_cluster_data()
            is_aggregated = meta.get('aggregated', False)
            k_clusters = meta.get('k', 3)

            # --- APLIKACE FILTRŮ ---
            if dd_resp_vis.value != 'Všichni':
                df_plot = df_plot[df_plot['participant'] == dd_resp_vis.value]
            if dd_exp_vis.value != 'Všechny':
                df_plot = df_plot[df_plot['Zkusenosti'] == dd_exp_vis.value]
            if not is_aggregated:
                if dd_stim_vis.value != 'Všechny':
                    df_plot = df_plot[df_plot['stimulus'] == dd_stim_vis.value]
                if dd_int_vis.value != 'Všechny':
                    val = int(dd_int_vis.value.split()[0])
                    df_plot = df_plot[df_plot['Num_Intervals'] == val]

            if df_plot.empty:
                print("Pro tuto kombinaci filtrů nezbyla žádná data k vykreslení.")
                return

            # --- SESTAVENÍ ČISTÉHO DYNAMICKÉHO TITULKU ---
            active_filters = []
            if dd_resp_vis.value != 'Všichni': active_filters.append(f"{dd_resp_vis.value}")
            if not is_aggregated and dd_stim_vis.value != 'Všechny': active_filters.append(f"{dd_stim_vis.value}")
            if dd_exp_vis.value != 'Všechny': active_filters.append(f"{dd_exp_vis.value}")
            if not is_aggregated and dd_int_vis.value != 'Všechny': active_filters.append(f"{dd_int_vis.value}")

            if is_aggregated:
                common_title = "Celkový profil respondentů" if not active_filters else " | ".join(active_filters)
            else:
                common_title = "Kompletní data" if not active_filters else " | ".join(active_filters)

            mode_str = "AOI přístup" if 'AOI' in dd_rqa_mode.value else "XY přístup"
            common_title = f"{mode_str} | {common_title}"

            df_plot_renamed = df_plot.rename(columns={
                'Zkusenosti': 'Zkušenost',
                'Cluster': 'Příslušnost ke shluku',
                'Num_Intervals': 'Počet intervalů legendy',
                'RR': 'Recurrence Rate'
            })

            fig, axes = plt.subplots(1, 3, figsize=(24, 7.5))
            real_palette = EXPERIENCE_COLORS if 'EXPERIENCE_COLORS' in globals() else {'Expert': '#1f77b4', 'Nováček': '#ff7f0e'}
            bubble_sizes = (40, 500)

            def setup_axis(ax, title):
                ax.set_title(title, fontsize=15, fontweight='bold', pad=15)
                ax.set_xlabel("DET", fontsize=12, fontweight='bold')
                ax.set_ylabel("LAM", fontsize=12, fontweight='bold')
                ax.set_xlim(-0.05, 1.05)
                ax.set_ylim(-0.05, 1.05)
                ax.grid(True, linestyle=':', alpha=0.6)

            # 1. Skutečné zkušenosti
            setup_axis(axes[0], f"1. Zkušenosti uživatelů\n({common_title})")
            if 'Zkušenost' in df_plot_renamed.columns and not df_plot_renamed['Zkušenost'].isna().all():
                sns.scatterplot(
                    data=df_plot_renamed, x='DET', y='LAM', hue='Zkušenost', size='Recurrence Rate',
                    sizes=bubble_sizes, size_norm=(global_rr_min, global_rr_max),
                    palette=real_palette, ax=axes[0], alpha=0.75, edgecolor='black', zorder=5
                )
                axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', labelspacing=1.4, borderpad=1)

            # 2. Vytvořené shluky
            setup_axis(axes[1], f"2. Shluky algoritmu K-means (k={k_clusters})\n({common_title})")
            cluster_palette = sns.color_palette('Set1', n_colors=k_clusters).as_hex()
            cluster_colors_dict = {f"Shluk {i+1}": cluster_palette[i] for i in range(k_clusters)}
            present_clusters = [f"Shluk {i+1}" for i in range(k_clusters) if f"Shluk {i+1}" in df_plot_renamed['Příslušnost ke shluku'].values]

            sns.scatterplot(
                data=df_plot_renamed, x='DET', y='LAM', hue='Příslušnost ke shluku', size='Recurrence Rate',
                sizes=bubble_sizes, size_norm=(global_rr_min, global_rr_max),
                palette=cluster_colors_dict, ax=axes[1], alpha=0.75, edgecolor='black', zorder=5, hue_order=present_clusters
            )
            axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', labelspacing=1.4, borderpad=1)

            # 3. Složitost legendy
            setup_axis(axes[2], f"3. Vliv složitosti legendy\n({common_title})")
            if not is_aggregated and 'Počet intervalů legendy' in df_plot_renamed.columns and df_plot_renamed['Počet intervalů legendy'].notna().any():
                intervals_order = sorted(df_plot_renamed['Počet intervalů legendy'].dropna().unique().tolist())
                sns.scatterplot(
                    data=df_plot_renamed, x='DET', y='LAM', hue='Počet intervalů legendy', size='Recurrence Rate',
                    sizes=bubble_sizes, size_norm=(global_rr_min, global_rr_max),
                    palette='viridis', ax=axes[2], alpha=0.75, edgecolor='black', zorder=5, hue_order=intervals_order
                )
                axes[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left', labelspacing=1.4, borderpad=1)
            else:
                info_text = "Data zprůměrována za uživatele.\nSložitost stimulů nelze zobrazit." if is_aggregated else "Údaje o počtu intervalů\nnejsou v datech k dispozici."
                axes[2].text(0.5, 0.5, info_text, ha='center', va='center', fontsize=12, style='italic', color='gray')

            plt.tight_layout()
            plt.show()

    btn_plot.on_click(on_plot_clicked)

    box_filters1 = widgets.HBox([dd_resp_vis, dd_stim_vis])
    box_filters2 = widgets.HBox([dd_exp_vis, dd_int_vis])
    display(widgets.VBox([lbl_filters, dd_rqa_mode, box_filters1, box_filters2, btn_plot, out_plot]))

In [ ]:
# @title Vizualizace s omezením počtu fixací v jednotlivých úkolech { display-mode: "form" }
# @markdown Tento blok kódu umožňuje interaktivně zkoumat, jak se mění vizuální a kognitivní strategie v závislosti na délce prohlížení (kognitivním úsilí - počtu fixací).
# @markdown * **Filtrace počtu fixací:** Pomocí posuvníku můžete omezit analyzovaný vzorek pouze na ty pokusy, které trvaly určitý počet fixací (např. pouze velmi krátká, nebo naopak velmi dlouhá měření).
# @markdown * **Horní panel (Bodové grafy):** Ukazuje vzájemné vztahy mezi metrikami (RR, DET, LAM). Body spadající do vámi zvoleného limitu fixací jsou barevně zvýrazněny podle zkušenosti respondentů. Hodnoty mimo tento limit zůstávají v pozadí jako šedé body pro zachování celkového kontextu.
# @markdown * **Spodní panel (Boxploty):** Detailně zobrazuje statistické rozložení metrik a přesné mediány výhradně pro data splňující zadaný limit fixací. Změnou posuvníku tak můžete dynamicky sledovat, jak se mění strategické chování (determinismus a laminarita) při narůstajícím čase stráveném nad mapou.
# @markdown * **Úroveň agregace:** Můžete analyzovat každý pokus samostatně, nebo nechat skript vypočítat průměrné hodnoty za celého respondenta a zkoumat tak jeho celkový zprůměrovaný profil.
# --- 1. NAČTĚNÍ DAT ---
folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
file_aoi = folder_path + 'rqa_results_AOI.csv'
file_xy = folder_path + 'rqa_results_XY.csv'
file_cleaned = folder_path + 'SegmentedData-cleaned.csv'

df_aoi = pd.read_csv(file_aoi) if os.path.exists(file_aoi) else pd.DataFrame()
df_xy = pd.read_csv(file_xy) if os.path.exists(file_xy) else pd.DataFrame()
df_raw = pd.read_csv(file_cleaned) if os.path.exists(file_cleaned) else pd.DataFrame()

# Výpočet fixací pro filtraci
fix_counts = pd.DataFrame()
max_fixations = 1000 # Výchozí hodnota
if not df_raw.empty:
    fix_counts = df_raw.groupby(['participant', 'stimulus']).size().reset_index(name='Pocet_fixaci')
    max_fixations = int(fix_counts['Pocet_fixaci'].max()) + 1

options = []
if not df_aoi.empty: options.append('AOI přístup')
if not df_xy.empty: options.append('XY přístup')

if 'EXPERIENCE_COLORS' not in globals():
    EXPERIENCE_COLORS = {'Nováček': '#ff7f0e', 'Expert': '#1f77b4'}

if not options:
    print("Nebyla nalezena žádná vypočítaná RQA data.")
else:
    # --- 2. UŽIVATELSKÉ ROZHRANÍ ---
    lbl_scatter = widgets.Label("Vizuální trendy strategií (Filtrace ovlivňuje barevné tečky a spodní boxploty):", layout=widgets.Layout(font_weight='bold', font_size='14px'))
    dd_mode_scatter = widgets.Dropdown(options=options, description='Režim RQA:', style={'description_width': 'initial'})
    dd_agg_scatter = widgets.Dropdown(
        options=[('1 tečka = 1 měření (Všechny pokusy)', 'raw'),
                 ('1 tečka = 1 participant (Průměrný profil)', 'agg')],
        description='Agregace:',
        style={'description_width': 'initial'}
    )
    slider_fixations = widgets.IntRangeSlider(
        value=[0, max_fixations],
        min=0,
        max=max_fixations,
        step=1,
        description='Počet fixací:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    btn_scatter = widgets.Button(description='Vykreslit grafy', button_style='primary', layout=widgets.Layout(width='220px', margin='10px 0 0 0'))
    out_scatter = widgets.Output()

    def on_scatter_clicked(b):
        with out_scatter:
            clear_output(wait=True)
            mode = dd_mode_scatter.value
            df = df_xy.copy() if 'XY' in mode else df_aoi.copy()

            if df.empty:
                print("Data pro tento režim chybí.")
                return

            df = df.dropna(subset=['RR', 'DET', 'LAM'])

            # Připojení počtu fixací pro filtraci
            if not fix_counts.empty:
                df = pd.merge(df, fix_counts, on=['participant', 'stimulus'], how='inner')
            else:
                df['Pocet_fixaci'] = 0

            # Aplikace agregace, pokud je zvolena (1 bod = 1 člověk)
            if dd_agg_scatter.value == 'agg':
                print("Provádím agregaci na úrovni respondentů (výpočet průměrných profilů)...")
                df = df.groupby(['participant', 'Zkusenosti']).agg({
                    'RR': 'mean',
                    'DET': 'mean',
                    'LAM': 'mean',
                    'Pocet_fixaci': 'mean'
                }).reset_index()

            # Rozdělení dat na "Aktivní" (v intervalu) a "Neaktivní"
            min_f, max_f = slider_fixations.value
            mask_in_range = (df['Pocet_fixaci'] >= min_f) & (df['Pocet_fixaci'] <= max_f)

            df_active = df[mask_in_range]
            df_inactive = df[~mask_in_range]

            # --- VYKRESLENÍ GRAFŮ (2 řady, 3 sloupce) ---
            fig, axes = plt.subplots(2, 3, figsize=(22, 13))
            agg_text = "Profil respondenta" if dd_agg_scatter.value == 'agg' else "Všechna měření"
            fig.suptitle(f"Závislost metrik s interaktivním filtrem počtu fixací ({mode} | {agg_text})", fontsize=20, fontweight='bold', y=0.98)

            has_exp = 'Zkusenosti' in df.columns
            hue_col = 'Zkusenosti' if has_exp else None
            palette = EXPERIENCE_COLORS if has_exp else None
            hue_order = sorted(df['Zkusenosti'].unique()) if has_exp else None

            # ---------------------------------------------------------
            # 1. ŘADA: SCATTER PLOTY
            # ---------------------------------------------------------
            # 1. GRAF: RR vs DET
            if not df_inactive.empty:
                sns.scatterplot(data=df_inactive, x='RR', y='DET', color='lightgray', alpha=0.4, s=60, edgecolor='darkgray', ax=axes[0, 0], legend=False)
            if not df_active.empty:
                sns.scatterplot(data=df_active, x='RR', y='DET', hue=hue_col, palette=palette, hue_order=hue_order, alpha=0.8, s=110, edgecolor='black', ax=axes[0, 0], legend=False)
            axes[0, 0].set_title("Vztah: DET a RR", fontsize=14, fontweight='bold')

            # 2. GRAF: RR vs LAM
            if not df_inactive.empty:
                sns.scatterplot(data=df_inactive, x='RR', y='LAM', color='lightgray', alpha=0.4, s=60, edgecolor='darkgray', ax=axes[0, 1], legend=False)
            if not df_active.empty:
                sns.scatterplot(data=df_active, x='RR', y='LAM', hue=hue_col, palette=palette, hue_order=hue_order, alpha=0.8, s=110, edgecolor='black', ax=axes[0, 1], legend=False)
            axes[0, 1].set_title("Vztah: LAM a RR)", fontsize=14, fontweight='bold')

            # 3. GRAF: DET vs LAM
            if not df_inactive.empty:
                sns.scatterplot(data=df_inactive, x='DET', y='LAM', color='lightgray', alpha=0.4, s=60, edgecolor='darkgray', ax=axes[0, 2], legend=False)
            if not df_active.empty:
                sns.scatterplot(data=df_active, x='DET', y='LAM', hue=hue_col, palette=palette, hue_order=hue_order, alpha=0.8, s=110, edgecolor='black', ax=axes[0, 2], legend=(True if has_exp else False))
            axes[0, 2].set_title("Vztah: LAM a DET", fontsize=14, fontweight='bold')

            for ax in axes[0]:
                ax.set_xlim(-0.05, 1.05)
                ax.set_ylim(-0.05, 1.05)
                ax.grid(True, linestyle=':', alpha=0.6)

            if has_exp and axes[0, 2].get_legend() is not None:
                axes[0, 2].legend(title='Skupina v limitu', bbox_to_anchor=(1.05, 1), loc='upper left')

            # ---------------------------------------------------------
            # 2. ŘADA: BOXPLOTY (Pojmou jen filtrovaná data = df_active)
            # ---------------------------------------------------------
            metrics = ['RR', 'DET', 'LAM']
            metric_titles = ['Recurrence Rate (RR)', 'Determinismus (DET)', 'Laminarita (LAM)']

            for i, m in enumerate(metrics):
                ax_box = axes[1, i]
                if not df_active.empty and has_exp:
                    sns.boxplot(data=df_active, x='Zkusenosti', y=m, palette=palette, order=hue_order, ax=ax_box, width=0.5, showfliers=False)

                    # Mediány
                    medians = df_active.groupby('Zkusenosti')[m].median()
                    for tick, label in enumerate(hue_order):
                        if pd.notna(medians.get(label)):
                            med_val = medians[label]
                            txt = ax_box.text(tick, med_val, f'{med_val:.3f}', ha='center', va='center', fontweight='bold', color='black', fontsize=11)
                            txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

                ax_box.set_title(f"Distribuce {m} s omezením počtu fixací", fontsize=13, fontweight='bold')
                ax_box.set_ylim(-0.05, 1.05)
                ax_box.set_xlabel('')
                ax_box.set_ylabel(metric_titles[i], fontsize=11)
                ax_box.grid(True, axis='y', linestyle=':', alpha=0.6)

            plt.subplots_adjust(hspace=0.35)
            plt.show()
            plt.close(fig) # Vyčištění paměti proti zdvojování grafů

            print("-" * 80)
            print(f"Zobrazeno {len(df_active)} z celkových {len(df)} záznamů (vyhovující filtru {min_f} - {max_f} fixací).")
            print("Spodní boxploty ukazují přesné rozložení metrik a mediány POUZE pro tyto barevně zvýrazněné body.")

    btn_scatter.on_click(on_scatter_clicked)

    # Skládání UI s posuvníkem
    ui_controls = widgets.VBox([
        lbl_scatter,
        widgets.HBox([dd_mode_scatter, dd_agg_scatter]),
        slider_fixations,
        btn_scatter
    ])
    display(widgets.VBox([ui_controls, out_scatter]))

In [ ]:
# @title Analýza reziduí { display-mode: "form" }
# @markdown Tento analytický nástroj provádí tzv. analýzu reziduí (odchylek od hlavního trendu). Zkoumá, zda odchýlení se od typické kognitivní strategie přináší respondentům výhodu (rychlejší řešení), nebo zda značí zmatek (nutnost delšího prohlížení).
# @markdown * **Horní panel (Regresní modely):** Skript vypočítá linku lineární regrese, která představuje "zlatý střed" chování napříč všemi uživateli. Body nad osou označují nadprůměrnou přítomnost dané metriky vůči trendu (např. extrémní zasekávání zraku), body pod osou naopak její podprůměrný výskyt.
# @markdown * **Spodní panel (Kognitivní náročnost):** Rozděluje data do dvou táborů přesně podle toho, na jaké straně trendové čáry se bod nacházel. Následně pomocí krabicových grafů vizualizuje, jaký dopad měla tato odchylka na absolutní počet fixací.
# @markdown * Tímto způsobem lze například exaktně dokázat, že ačkoliv dva lidé mají stejnou hodnotu Determinismu, ten, kdo u toho vykazuje nadprůměrnou Laminaritu (častější vracení se do stejných míst - je "nad trendem"), spotřebuje pro vyřešení stejného úkolu dvojnásobné množství fixací.
# --- 1. PŘÍPRAVA DAT ---
folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
file_cleaned = folder_path + 'SegmentedData-cleaned.csv'
df_raw = pd.read_csv(file_cleaned) if os.path.exists(file_cleaned) else pd.DataFrame()

if 'EXPERIENCE_COLORS' not in globals():
    EXPERIENCE_COLORS = {'Nováček': '#ff7f0e', 'Expert': '#1f77b4'}

# --- 2. UI ---
lbl_resid = widgets.Label("Finální analýza: Čisté vizualizace strategií pro bakalářskou práci", layout=widgets.Layout(font_weight='bold'))
dd_mode_resid = widgets.Dropdown(options=['Prostorový (XY)', 'Sémantický (AOI)'], description='Data z:')
dd_agg_mode = widgets.Dropdown(
    options=[('1 tečka = 1 měření (Všechny pokusy)', 'raw'),
             ('1 tečka = 1 participant (Průměrný profil)', 'agg')],
    description='Agregace:',
    style={'description_width': 'initial'}
)
btn_triple = widgets.Button(description='Generovat grafy', button_style='success', layout=widgets.Layout(width='200px'))
out_triple = widgets.Output()

def run_triple_analysis(b):
    with out_triple:
        clear_output(wait=True)

        mode_file = 'rqa_results_XY.csv' if 'XY' in dd_mode_resid.value else 'rqa_results_AOI.csv'
        if not os.path.exists(folder_path + mode_file):
            print(f"Chyba: Soubor {mode_file} nebyl nalezen.")
            return

        df_rqa = pd.read_csv(folder_path + mode_file).dropna(subset=['RR', 'DET', 'LAM'])
        fix_counts = df_raw.groupby(['participant', 'stimulus', 'Zkusenosti']).size().reset_index(name='Pocet_fixaci')
        df_full = pd.merge(df_rqa, fix_counts, on=['participant', 'stimulus', 'Zkusenosti'])

        if dd_agg_mode.value == 'agg':
            df_full = df_full.groupby(['participant', 'Zkusenosti']).agg({
                'RR': 'mean', 'DET': 'mean', 'LAM': 'mean', 'Pocet_fixaci': 'mean'
            }).reset_index()

        # DYNAMICKÉ SJEDNOCENÍ OS PRO RQA (HORNÍ ŘADA)
        all_rqa_vals = pd.concat([df_full['RR'], df_full['DET'], df_full['LAM']])
        rqa_min, rqa_max = all_rqa_vals.min() * 0.9, all_rqa_vals.max() * 1.1

        # DYNAMICKÉ SJEDNOCENÍ OS PRO FIXACE (DOLNÍ ŘADA)
        fix_min, fix_max = df_full['Pocet_fixaci'].min() * 0.9, df_full['Pocet_fixaci'].max() * 1.1

        config = [
            ('RR', 'DET', 'Recurrence Rate (RR)', 'Determinismus (DET)', 'Vztah: DET a RR'),
            ('RR', 'LAM', 'Recurrence Rate (RR)', 'Laminarita (LAM)', 'Vztah: LAM a RR'),
            ('DET', 'LAM', 'Determinismus (DET)', 'Laminarita (LAM)', 'Vztah: LAM a DET')
        ]

        fig, axes = plt.subplots(2, 3, figsize=(22, 13))
        agg_text = "Profil respondenta" if dd_agg_mode.value == 'agg' else "Jednotlivá měření"
        fig.suptitle(f"Strategie participantů – porovnání ({dd_mode_resid.value} | {agg_text})", fontsize=22, fontweight='bold', y=0.98)

        group_order = ['Nad trendem', 'Pod trendem']
        hue_order = sorted(df_full['Zkusenosti'].unique())

        handles_strat, labels_strat = None, None

        for i, (col_x, col_y, label_x, label_y, title) in enumerate(config):
            df = df_full.copy()
            slope, intercept, r_val, p_val_corr, std_err = linregress(df[col_x], df[col_y])
            df['Ocekavane'] = slope * df[col_x] + intercept
            df['Skupina'] = np.where(df[col_y] >= df['Ocekavane'], 'Nad trendem', 'Pod trendem')

            # 1. ŘADA: SCATTER PLOTY (Zbarveno dle Zkušenosti)
            ax_scat = axes[0, i]
            sns.scatterplot(data=df, x=col_x, y=col_y, hue='Zkusenosti', palette=EXPERIENCE_COLORS, s=110, alpha=0.65, ax=ax_scat, edgecolor='black', hue_order=hue_order)

            x_line = np.array([df[col_x].min(), df[col_x].max()])
            ax_scat.plot(x_line, slope * x_line + intercept, color='black', linestyle='--', linewidth=2.5)

            ax_scat.set_title(title, fontsize=15, fontweight='bold', pad=15)
            ax_scat.set_xlabel(label_x, fontsize=12)
            ax_scat.set_ylabel(label_y, fontsize=12)
            ax_scat.set_xlim(rqa_min, rqa_max)
            ax_scat.set_ylim(rqa_min, rqa_max)

            # Záchrana popisků pro centrální legendu z prvního grafu
            if i == 0:
                handles_strat, labels_strat = ax_scat.get_legend_handles_labels()

            ax_scat.get_legend().remove()
            ax_scat.grid(True, linestyle=':', alpha=0.6)

            # 2. ŘADA: ČISTÉ BOXPLOTY (Bez teček stripplotu)
            ax_box = axes[1, i]
            sns.boxplot(data=df, x='Skupina', y='Pocet_fixaci', hue='Zkusenosti', palette=EXPERIENCE_COLORS,
                        ax=ax_box, width=0.6, order=group_order, hue_order=hue_order,
                        showfliers=False, flierprops={"marker": "x", "markersize": 5})

            x_labels = []
            for grp in group_order:
                n_exp = len(df[(df['Skupina'] == grp) & (df['Zkusenosti'] == 'Expert')])
                n_nov = len(df[(df['Skupina'] == grp) & (df['Zkusenosti'] == 'Nováček')])
                x_labels.append(f"{grp}\n(E: {n_exp}, N: {n_nov})")

            ax_box.set_xticklabels(x_labels, fontsize=11, fontweight='bold')
            ax_box.set_ylim(fix_min, fix_max)

            # Mediány s path_effects pro čitelnost
            for x_idx, grp_name in enumerate(group_order):
                for h_idx, hue_name in enumerate(hue_order):
                    subset = df[(df['Skupina'] == grp_name) & (df['Zkusenosti'] == hue_name)]
                    if not subset.empty:
                        med_val = subset['Pocet_fixaci'].median()
                        offset = (h_idx - (len(hue_order)-1)/2) * (0.6 / len(hue_order))
                        txt = ax_box.text(x_idx + offset, med_val, f'{med_val:.1f}',
                                          ha='center', va='center', fontweight='bold', color='black', fontsize=11)
                        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

            ax_box.set_title(f"Kognitivní náročnost (počet fixací)", fontsize=13, fontweight='bold')
            ax_box.set_ylabel('Počet fixací', fontsize=12)
            ax_box.set_xlabel('')
            ax_box.grid(True, axis='y', linestyle=':', alpha=0.6)

            # Odstranění legend z boxplotů (vše řeší centrální legenda)
            if ax_box.get_legend() is not None:
                ax_box.get_legend().remove()

        # Legenda uprostřed (Platí pro scatter i boxplot)
        fig.legend(handles_strat, labels_strat, loc='center', bbox_to_anchor=(0.5, 0.52),
                         ncol=2, title="Zkušenost respondenta", fontsize=12, title_fontsize=13, frameon=True, shadow=True)

        plt.subplots_adjust(hspace=0.45)
        plt.show()

btn_triple.on_click(run_triple_analysis)
display(widgets.VBox([lbl_resid, widgets.HBox([dd_mode_resid, dd_agg_mode]), btn_triple, out_triple]))

In [ ]:
# @title Závislost RQA metrik na počtu fixací { display-mode: "form" }
# @markdown Tento analytický blok zkoumá přímou lineární závislost (korelaci) mezi celkovým kognitivním úsilím (množstvím fixací) a tvarem vizuální strategie (hodnotami RQA metrik).
# @markdown * **Režim a agregace:** Zvolte si matematický přístup (AOI/XY) a úroveň detailu (všechna jednotlivá měření vs. zprůměrované profily respondentů).
# @markdown * **Bodové grafy:** Ukazují rozložení dat na ose X (celkový počet fixací v mapě) a ose Y (konkrétní RQA metrika). Body jsou barevně odlišeny podle úrovně zkušeností.
# @markdown * **Regresní trend:** Skript napříč všemi body proloží globální trendovou čáru. Ta graficky znázorňuje obecný vývoj strategie při narůstajícím kognitivním úsilí bez ohledu na to, zda mapu četl expert nebo nováček.
# @markdown * **Statistické testování:** V titulku každého ze tří grafů se automaticky vypočítá Pearsonův korelační koeficient (r) a vyhodnotí se jeho statistická významnost. Exaktně tak zjistíte, zda například delší sledování mapy přirozeně vede k vyšší míře lokálního zasekávání zraku (LAM), nebo zda jsou vizuální strategie na délce prohlížení zcela nezávislé.
# --- 1. NAČTĚNÍ DAT ---
folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
file_aoi = folder_path + 'rqa_results_AOI.csv'
file_xy = folder_path + 'rqa_results_XY.csv'
file_cleaned = folder_path + 'SegmentedData-complete.csv'

df_aoi = pd.read_csv(file_aoi) if os.path.exists(file_aoi) else pd.DataFrame()
df_xy = pd.read_csv(file_xy) if os.path.exists(file_xy) else pd.DataFrame()
df_raw = pd.read_csv(file_cleaned) if os.path.exists(file_cleaned) else pd.DataFrame()

options = []
if not df_aoi.empty: options.append('Přístup AOI')
if not df_xy.empty: options.append('Přístup XY')

if 'EXPERIENCE_COLORS' not in globals():
    EXPERIENCE_COLORS = {'Nováček': '#ff7f0e', 'Expert': '#1f77b4'}

if not options or df_raw.empty:
    print("Data chybí. Zkontrolujte prosím cestu k souborům s RQA výsledky a očištěnými fixacemi.")
else:
    # Výpočet počtu fixací z původních očištěných dat
    fix_counts = df_raw.groupby(['participant', 'stimulus', 'Zkusenosti']).size().reset_index(name='Pocet_fixaci')

    # --- 2. UŽIVATELSKÉ ROZHRANÍ ---
    lbl_title = widgets.Label("Závislost metrik na počtu fixací:", layout=widgets.Layout(font_weight='bold', font_size='14px'))
    dd_mode = widgets.Dropdown(options=options, description='Režim RQA:', style={'description_width': 'initial'})
    dd_agg = widgets.Dropdown(
        options=[('1 tečka = 1 měření (Všechny pokusy)', 'raw'),
                 ('1 tečka = 1 participant (Průměrný profil)', 'agg')],
        description='Agregace:', style={'description_width': 'initial'}
    )
    btn_plot = widgets.Button(description='Vykreslit závislost', button_style='primary', layout=widgets.Layout(width='200px', margin='10px 0 0 0'))
    out_plot = widgets.Output()

    def draw_plots(b):
        with out_plot:
            clear_output(wait=True)
            mode = dd_mode.value
            df_rqa = df_xy.copy() if 'XY' in mode else df_aoi.copy()

            if df_rqa.empty:
                print("Data pro tento režim chybí.")
                return

            df_rqa = df_rqa.dropna(subset=['RR', 'DET', 'LAM'])

            # Sloučení RQA výsledků s celkovým počtem fixací pro daný pokus
            if 'Zkusenosti' in df_rqa.columns:
                df = pd.merge(df_rqa, fix_counts[['participant', 'stimulus', 'Pocet_fixaci']], on=['participant', 'stimulus'], how='inner')
            else:
                df = pd.merge(df_rqa, fix_counts, on=['participant', 'stimulus'], how='inner')

            if dd_agg.value == 'agg':
                df = df.groupby(['participant', 'Zkusenosti']).agg({
                    'RR': 'mean', 'DET': 'mean', 'LAM': 'mean', 'Pocet_fixaci': 'mean'
                }).reset_index()

            fig, axes = plt.subplots(1, 3, figsize=(22, 7))
            agg_text = "Průměrné profily respondentů" if dd_agg.value == 'agg' else "Všechna měření samostatně"
            fig.suptitle(f"Vliv počtu fixací na jednotlivé metriky ({mode} | {agg_text})", fontsize=20, fontweight='bold', y=1.05)

            metrics = [('RR', 'Recurrence Rate (RR)'), ('DET', 'Determinismus (DET)'), ('LAM', 'Laminarita (LAM)')]
            hue_order = sorted(df['Zkusenosti'].unique())

            for i, (col_y, label_y) in enumerate(metrics):
                ax = axes[i]

                # 1. Body grafu obarvené podle zkušenosti
                sns.scatterplot(data=df, x='Pocet_fixaci', y=col_y, hue='Zkusenosti', palette=EXPERIENCE_COLORS,
                                hue_order=hue_order, s=110, alpha=0.75, edgecolor='black', ax=ax)

                # 2. Celkový regresní trend (prokazující společnou závislost bez ohledu na zkušenost)
                sns.regplot(data=df, x='Pocet_fixaci', y=col_y, scatter=False, color='black', ci=None,
                            line_kws={'linestyle':'--', 'linewidth':3}, ax=ax)

                # Výpočet statistické závislosti (Pearson)
                slope, intercept, r_val, p_val, std_err = linregress(df['Pocet_fixaci'], df[col_y])
                significance = "významné (*)" if p_val < 0.05 else "nevýznamné (ns)"

                ax.set_title(f"Vliv počtu fixací na {col_y}\n(Korelace r = {r_val:.2f}, {significance})", fontsize=14, fontweight='bold', pad=15)
                ax.set_ylabel(label_y, fontsize=12, fontweight='bold')
                ax.set_xlabel("Celkový počet fixací v mapě", fontsize=12, fontweight='bold')

                # Fixní osa Y od nuly pro RQA
                ax.set_ylim(-0.05, 1.05)

                # Dynamická osa X s mírným odsazením
                min_f, max_f = df['Pocet_fixaci'].min(), df['Pocet_fixaci'].max()
                padding = (max_f - min_f) * 0.05 if max_f > min_f else 10
                ax.set_xlim(max(0, min_f - padding), max_f + padding)

                ax.grid(True, linestyle=':', alpha=0.6)

                # Legenda pouze u posledního grafu
                if i != 2:
                    if ax.get_legend() is not None:
                        ax.get_legend().remove()
                else:
                    ax.legend(title='Skupina', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12, title_fontsize=13)

            plt.tight_layout()
            plt.show()

    btn_plot.on_click(draw_plots)

    ui_controls = widgets.VBox([lbl_title, widgets.HBox([dd_mode, dd_agg]), btn_plot])
    display(widgets.VBox([ui_controls, out_plot]))

In [ ]:
# @title Porovnání metrik uvnitř zón (Mapa a Legenda){ display-mode: "form" }
# @markdown Tento blok se zaměřuje na strukturální analýzu uvnitř stimulu a porovnává kognitivní chování respondentů odděleně pro dvě hlavní sémantické zóny: samotné mapové pole a mapový klíč (legendu).
# @markdown * **Dynamický výpočet:** Algoritmus izoluje fixace ležící v mapě od fixací spadajících do legendy. Následně pro obě tyto subsekvence zcela nezávisle spočítá RQA metriky (volitelně sémantickým nebo prostorovým přístupem).
# @markdown * **Horní panel (Statistika a distribuce):** Krabicové grafy porovnávají RQA metriky (RR, DET, LAM) a celkové kognitivní úsilí (absolutní počet fixací) mezi zónami. Skript navíc na pozadí automaticky aplikuje Mann-Whitney U test a v titulcích grafů rovnou vyhodnocuje statistickou významnost (p-hodnotu) nalezených rozdílů.
# @markdown * **Spodní panel (Vzájemný poměr strategií):** Bodový graf graficky vizualizuje strukturální odlišnosti.
# --- 1. NAČTĚNÍ DAT A PŘÍPRAVA ---
try:
    folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    file_cleaned = FILE_COMPLETE_CSV if 'FILE_COMPLETE_CSV' in globals() else folder_path + 'SegmentedData-cleaned.csv'
    file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
except NameError:
    folder_path = "drive/MyDrive/BAK-RQA/"
    file_cleaned = folder_path + 'SegmentedData-cleaned.csv'
    file_rqa_xy = folder_path + 'rqa_results_XY.csv'

try:
    df_cleaned = pd.read_csv(file_cleaned, low_memory=False)
    available_stimuli = ['Všechny'] + sorted(df_cleaned['stimulus'].dropna().unique().tolist())
except Exception as e:
    print("Chyba: Očištěná data nebyla nalezena.")
    df_cleaned = pd.DataFrame()
    available_stimuli = ['Všechny']

# Zjištění radiusu pro XY
default_radius = 50
if os.path.exists(file_rqa_xy):
    try:
        df_xy_temp = pd.read_csv(file_rqa_xy)
        if 'Radius_px' in df_xy_temp.columns:
            default_radius = int(df_xy_temp['Radius_px'].iloc[0])
    except: pass

rqa_l_min = RQA_L_MIN if 'RQA_L_MIN' in globals() else 2
rqa_v_min = RQA_V_MIN if 'RQA_V_MIN' in globals() else 2

if 'EXPERIENCE_COLORS' not in globals():
    EXPERIENCE_COLORS = {'Nováček': '#ff7f0e', 'Expert': '#1f77b4'}

# --- 2. UŽIVATELSKÉ ROZHRANÍ ---
lbl_scenario = widgets.Label("Komplexní porovnání vnitřních strategií: Mapové pole vs. Legenda:", layout=widgets.Layout(font_weight='bold', font_size='14px'))
dd_rqa_mode = widgets.Dropdown(options=['AOI přístup', 'XY přístup'], description='Režim RQA:', style={'description_width': 'initial'})
dd_stimulus = widgets.Dropdown(options=available_stimuli, description='Stimul:', style={'description_width': 'initial'})

btn_calc_plot = widgets.Button(description='Vykreslit analýzu', button_style='success', layout=widgets.Layout(width='200px', margin='10px 0 0 0'))
out_plot = widgets.Output()

# --- 3. MATEMATICKÉ FUNKCE ---
def get_line_lengths(arr, min_length=2):
    pad = np.pad(arr, (1, 1), mode='constant', constant_values=0)
    diffs = np.diff(pad)
    starts = np.where(diffs == 1)[0]
    ends = np.where(diffs == -1)[0]
    lengths = ends - starts
    return lengths[lengths >= min_length]

def extract_rqa_from_rp(RP, N, l_min, v_min):
    recurrent_points = np.sum(RP)
    total_possible = N * (N - 1)
    RR = recurrent_points / total_possible if total_possible > 0 else 0
    diag_lines = []
    for d in range(1, N):
        diag = np.diagonal(RP, offset=d)
        diag_lines.extend(get_line_lengths(diag, l_min))
    DET = np.sum(diag_lines) / (recurrent_points / 2) if recurrent_points > 0 and len(diag_lines) > 0 else 0
    vert_lines = []
    for c in range(N):
        col = RP[:, c]
        vert_lines.extend(get_line_lengths(col, v_min))
    LAM = np.sum(vert_lines) / recurrent_points if recurrent_points > 0 and len(vert_lines) > 0 else 0
    return pd.Series({'RR': RR, 'DET': DET, 'LAM': LAM})

def calculate_rqa_aoi(seq, l_min, v_min):
    seq = np.array(seq)
    N = len(seq)
    if N < 5: return pd.Series({'RR': np.nan, 'DET': np.nan, 'LAM': np.nan, 'Fixaci': N})
    RP = (seq[:, None] == seq[None, :]).astype(int)
    np.fill_diagonal(RP, 0)
    res = extract_rqa_from_rp(RP, N, l_min, v_min)
    res['Fixaci'] = N
    return res

def calculate_rqa_spatial(x_coords, y_coords, radius, l_min, v_min):
    coords = np.column_stack((x_coords, y_coords))
    N = len(coords)
    if N < 5: return pd.Series({'RR': np.nan, 'DET': np.nan, 'LAM': np.nan, 'Fixaci': N})
    dist_matrix = cdist(coords, coords, metric='euclidean')
    RP = (dist_matrix <= radius).astype(int)
    np.fill_diagonal(RP, 0)
    res = extract_rqa_from_rp(RP, N, l_min, v_min)
    res['Fixaci'] = N
    return res

# --- 4. HLAVNÍ VÝPOČET A VYKRESLENÍ ---
def on_calc_plot_clicked(b):
    with out_plot:
        clear_output(wait=True)
        if df_cleaned.empty: return

        is_xy_mode = 'XY' in dd_rqa_mode.value
        selected_stim = dd_stimulus.value

        df_work = df_cleaned.copy()
        if selected_stim != 'Všechny':
            df_work = df_work[df_work['stimulus'] == selected_stim]

        df_work['AOI Single'] = df_work['AOI Single'].astype(str)
        legend_and_meta = [str(i) for i in range(1, 21)] + ['Title', 'Imprint']

        # Rozdělení oblastí
        df_map = df_work[~df_work['AOI Single'].isin(legend_and_meta)].copy()
        legend_only = [str(i) for i in range(1, 21)]
        df_leg = df_work[df_work['AOI Single'].isin(legend_only)].copy()

        grouping_cols = ['participant', 'stimulus', 'Zkusenosti']
        results_list = []

        print(f"Zpracovávám dynamické sekvence a počítám statistiku pro obě oblasti...")

        for label, df_sub in [('Mapové pole', df_map), ('Legenda', df_leg)]:
            if not df_sub.empty:
                if is_xy_mode:
                    df_sub['x'] = pd.to_numeric(df_sub['x'], errors='coerce')
                    df_sub['y'] = pd.to_numeric(df_sub['y'], errors='coerce')
                    df_sub = df_sub.dropna(subset=['x', 'y'])
                    res = df_sub.groupby(grouping_cols).apply(lambda g: calculate_rqa_spatial(g['x'].values, g['y'].values, default_radius, rqa_l_min, rqa_v_min)).reset_index()
                else:
                    res = df_sub.groupby(grouping_cols).apply(lambda g: calculate_rqa_aoi(g['AOI Single'].values, rqa_l_min, rqa_v_min)).reset_index()
                res['Oblast'] = label
                results_list.append(res)

        df_results = pd.concat(results_list, ignore_index=True)
        df_results = df_results.dropna(subset=['RR', 'DET', 'LAM'])

        if df_results.empty:
            print("Při daných parametrech (minimum 5 fixací v zóně) nevznikla žádná platná data pro porovnání.")
            return

        if selected_stim == 'Všechny':
            df_results = df_results.groupby(['participant', 'Zkusenosti', 'Oblast'])[['RR', 'DET', 'LAM', 'Fixaci']].mean().reset_index()

        # --- VIZUALIZACE ---
        fig = plt.figure(figsize=(20, 14))
        gs = fig.add_gridspec(2, 4)

        mode_str = "XY přístup" if is_xy_mode else "AOI přístup"
        fig.suptitle(f"Vnitřní struktura Mapy vs. Legendy ({mode_str}) | Stimul: {selected_stim}", fontsize=20, fontweight='bold', y=0.98)

        metrics = ['RR', 'DET', 'LAM']
        axis_titles = ['Recurrence Rate (RR)', 'Determinismus (DET)', 'Laminarita (LAM)']
        oblast_order = ['Mapové pole', 'Legenda']
        group_order = sorted(df_results['Zkusenosti'].dropna().unique().tolist())

        # 1. ŘADA: Boxploty metrik + Statistika
        for i, metric in enumerate(metrics):
            ax = fig.add_subplot(gs[0, i])
            sns.boxplot(data=df_results, x='Oblast', y=metric, hue='Zkusenosti', ax=ax, palette=EXPERIENCE_COLORS, order=oblast_order, hue_order=group_order, width=0.6)

            # STATISTICKÉ OVĚŘENÍ ROZDÍLU (Mapa vs Legenda obecně)
            try:
                map_data = df_results[df_results['Oblast'] == 'Mapové pole'][metric].dropna()
                leg_data = df_results[df_results['Oblast'] == 'Legenda'][metric].dropna()
                stat, p_val = mannwhitneyu(map_data, leg_data, alternative='two-sided')
                sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns"))
                ax.set_title(f"{axis_titles[i]}\n(Rozdíl zón: p={p_val:.3f} {sig})", fontsize=13, fontweight='bold')
            except:
                ax.set_title(axis_titles[i], fontsize=13, fontweight='bold')

            ax.set_ylim(-0.05, 1.05)
            ax.set_xlabel('')
            ax.grid(True, linestyle=':', alpha=0.6)

            # Mediány
            for x_idx, label in enumerate(oblast_order):
                for h_idx, group in enumerate(group_order):
                    offset = (h_idx - (len(group_order)-1)/2) * (0.6/len(group_order))
                    val = df_results[(df_results['Oblast']==label) & (df_results['Zkusenosti']==group)][metric].median()
                    if pd.notna(val):
                        txt = ax.text(x_idx + offset, val, f'{val:.2f}', ha='center', va='center', fontweight='bold', color='black')
                        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

            ax.get_legend().remove()

        # Počet fixací (Kognitivní úsilí)
        ax_fix = fig.add_subplot(gs[0, 3])
        sns.boxplot(data=df_results, x='Oblast', y='Fixaci', hue='Zkusenosti', ax=ax_fix, palette=EXPERIENCE_COLORS, order=oblast_order, hue_order=group_order, width=0.6)
        ax_fix.set_title("Kognitivní úsilí (Počet fixací)", fontsize=13, fontweight='bold')
        ax_fix.set_xlabel('')
        ax_fix.set_ylabel('Absolutní počet fixací')
        ax_fix.grid(True, linestyle=':', alpha=0.6)

        # Mediány pro Fixaci
        for x_idx, label in enumerate(oblast_order):
            for h_idx, group in enumerate(group_order):
                offset = (h_idx - (len(group_order)-1)/2) * (0.6/len(group_order))
                val = df_results[(df_results['Oblast']==label) & (df_results['Zkusenosti']==group)]['Fixaci'].median()
                if pd.notna(val):
                    txt = ax_fix.text(x_idx + offset, val, f'{val:.0f}', ha='center', va='center', fontweight='bold', color='black')
                    txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

        ax_fix.legend(title='Zkušenost', bbox_to_anchor=(1.05, 1), loc='upper left')

        # 2. ŘADA: Scatter Plot (DET vs LAM)
        ax_scat = fig.add_subplot(gs[1, :])
        rr_min, rr_max = df_results['RR'].min(), df_results['RR'].max()

        df_plot = df_results.rename(columns={'RR': 'Recurrence Rate (RR)', 'Oblast': 'Sledovaná oblast'})
        area_palette = {'Mapové pole': '#d95f02', 'Legenda': '#7570b3'}

        sns.scatterplot(
            data=df_plot, x='DET', y='LAM', hue='Sledovaná oblast', size='Recurrence Rate (RR)',
            sizes=(50, 600), size_norm=(rr_min, rr_max), palette=area_palette,
            alpha=0.7, edgecolor='black', ax=ax_scat
        )

        ax_scat.set_title("Vzájemný poměr kognitivních strategií (DET vs LAM)", fontsize=15, fontweight='bold', pad=10)
        ax_scat.set_xlim(-0.05, 1.05); ax_scat.set_ylim(-0.05, 1.05)
        ax_scat.set_xlabel("Determinismus (DET) - Systematičnost", fontsize=12)
        ax_scat.set_ylabel("Laminarita (LAM) - Zasekávání / Dekódování", fontsize=12)
        ax_scat.grid(True, linestyle=':', alpha=0.6)
        ax_scat.legend(bbox_to_anchor=(1.02, 1), loc='upper left', labelspacing=1.2, borderaxespad=0.)

        plt.tight_layout()
        plt.show()

btn_calc_plot.on_click(on_calc_plot_clicked)
display(widgets.VBox([lbl_scenario, widgets.HBox([dd_rqa_mode, dd_stimulus]), btn_calc_plot, out_plot]))

In [ ]:
# @title Vývoj metrik v čase (Running RQA) { display-mode: "form" }
# @markdown Tento blok kódu umožňuje provést dynamickou analýzu v čase (tzv. Running RQA). Namísto jedné finální hodnoty pro celý pokus ukazuje, jak se kognitivní strategie vyvíjela krok za krokem, fixaci po fixaci.
# @markdown * **Parametry analýzy:** Vyberte přístup (AOI/XY), konkrétního respondenta a mapu. Nastavte velikost počátečního okna (od kolikáté fixace má výpočet začít) a prahovou hodnotu pro detekci významných skoků ve strategii.
# @markdown * **Princip výpočtu:** Algoritmus vezme počáteční sekvenci fixací a spočítá pro ni RQA matici. Poté přidá další fixaci v pořadí, matici přepočítá a tento proces opakuje až do konce pokusu. Tím vzniká časová řada postupného vývoje jednotlivých metrik.
# @markdown * **Interpretace grafů:** Výsledkem jsou tři pod sebou seřazené grafy (pro RR, DET a LAM). Skript automaticky detekuje a barevně označuje okamžiky, kdy přidání jediné fixace způsobilo prudkou změnu celkové metriky (např. náhlý nárůst zasekávání a zmatku).
# @markdown * **Podbarvení (pouze AOI):** Při volbě AOI přístupu je pozadí grafů navíc svisle podbarveno podle toho, na jakou oblast (AOI) se respondent v daném kroku díval. Lze tak přesně odhalit, který vizuální prvek na mapě byl spouštěčem změny uživatelova chování.
# --- 1. NAČTĚNÍ DAT ---
try:
    folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    file_cleaned = FILE_COMPLETE_CSV if 'FILE_COMPLETE_CSV' in globals() else folder_path + 'SegmentedData-cleaned.csv'
except NameError:
    folder_path = "drive/MyDrive/BAK-RQA/"
    file_cleaned = folder_path + 'SegmentedData-cleaned.csv'

try:
    df_cleaned = pd.read_csv(file_cleaned, low_memory=False)
    available_stimuli = sorted(df_cleaned['stimulus'].dropna().unique().tolist())
    initial_stimulus = available_stimuli[0] if available_stimuli else None

    if initial_stimulus:
        available_participants = sorted(df_cleaned[df_cleaned['stimulus'] == initial_stimulus]['participant'].unique().tolist())
    else:
        available_participants = []
except Exception as e:
    print("Chyba: Očištěná data nebyla nalezena. Zkontrolujte prosím cestu k souboru.")
    df_cleaned = pd.DataFrame()
    available_stimuli = []
    available_participants = []

# Kontrola, jaká RQA data vůbec máme pro zjištění rádiusu v případě XY
file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
df_rqa_xy = pd.read_csv(file_rqa_xy) if os.path.exists(file_rqa_xy) else pd.DataFrame()

# --- 2. UŽIVATELSKÉ ROZHRANÍ ---
lbl_running = widgets.Label("Nastavení pro Running RQA (vývoj metrik s narůstajícím oknem):", layout=widgets.Layout(font_weight='bold', font_size='14px'))

options_mode = ['přístup AOI']
if not df_rqa_xy.empty: options_mode.append('Přístup XY')
dd_rqa_mode = widgets.Dropdown(options=options_mode, description='Režim RQA:', style={'description_width': 'initial'})

dd_stimulus = widgets.Dropdown(options=available_stimuli, description='Stimul:', style={'description_width': 'initial'})
dd_participant = widgets.Dropdown(options=available_participants, description='Respondent:', style={'description_width': 'initial'})
min_window_slider = widgets.IntSlider(value=1, min=1, max=30, description='Počáteční fixace:', style={'description_width': 'initial'})
jump_threshold_slider = widgets.FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description='Zobrazit skoky větší než:', style={'description_width': 'initial'})

btn_run = widgets.Button(description='Vykreslit vývoj', button_style='success', layout=widgets.Layout(width='200px', margin='10px 0px 0px 0px'))
out_run = widgets.Output()

def update_participants(*args):
    if not df_cleaned.empty:
        selected_stim = dd_stimulus.value
        valid_parts = sorted(df_cleaned[df_cleaned['stimulus'] == selected_stim]['participant'].unique().tolist())
        dd_participant.options = valid_parts

dd_stimulus.observe(update_participants, 'value')


# --- 3. POMOCNÉ MATEMATICKÉ FUNKCE ---
def get_line_lengths(arr, min_length=2):
    pad = np.pad(arr, (1, 1), mode='constant', constant_values=0)
    diffs = np.diff(pad)
    starts = np.where(diffs == 1)[0]
    ends = np.where(diffs == -1)[0]
    lengths = ends - starts
    return lengths[lengths >= min_length]

def calculate_rqa_from_matrix(RP, N, l_min=2, v_min=2):
    recurrent_points = np.sum(RP)
    total_possible = N * (N - 1)
    RR = recurrent_points / total_possible if total_possible > 0 else 0

    diag_lines = []
    for d in range(1, N):
        diag = np.diagonal(RP, offset=d)
        diag_lines.extend(get_line_lengths(diag, l_min))
    DET = np.sum(diag_lines) / (recurrent_points / 2) if recurrent_points > 0 and len(diag_lines) > 0 else 0

    vert_lines = []
    for c in range(N):
        col = RP[:, c]
        vert_lines.extend(get_line_lengths(col, v_min))
    LAM = np.sum(vert_lines) / recurrent_points if recurrent_points > 0 and len(vert_lines) > 0 else 0

    return {'RR': RR, 'DET': DET, 'LAM': LAM}


# --- 4. HLAVNÍ FUNKCE PRO VÝPOČET A VYKRESLENÍ ---
def on_run_clicked(b):
    with out_run:
        clear_output(wait=True)
        stim = dd_stimulus.value
        part = dd_participant.value
        min_w = min_window_slider.value
        threshold = jump_threshold_slider.value
        is_xy_mode = 'XY' in dd_rqa_mode.value

        subset = df_cleaned[(df_cleaned['stimulus'] == stim) & (df_cleaned['participant'] == part)].sort_values('timestamp').copy()

        if subset.empty:
            print("Pro tuto kombinaci nejsou k dispozici žádná data.")
            return

        exp = subset['Zkusenosti'].iloc[0] if 'Zkusenosti' in subset.columns and not subset['Zkusenosti'].empty else 'Neznámá'

        N = len(subset)
        if N < min_w:
            print(f"Upozornění: Respondent má pouze {N} fixací, ale počáteční okno je {min_w}.")
            return

        print(f"Zpracovávám dynamický vývoj pro: {part} (Skupina: {exp}, Stimul: {stim}, Celkem fixací: {N})...")

        if is_xy_mode:
            subset['x'] = pd.to_numeric(subset['x'], errors='coerce')
            subset['y'] = pd.to_numeric(subset['y'], errors='coerce')
            coords = np.column_stack((subset['x'], subset['y']))
            current_radius = df_rqa_xy['Radius_px'].iloc[0] if not df_rqa_xy.empty and 'Radius_px' in df_rqa_xy.columns else 50
        else:
            subset['AOI Single'] = subset['AOI Single'].astype(str)
            sequence = subset['AOI Single'].tolist()

            AOI_COLORS = {
                'Brightest': '#7c68f4', 'Darkest': '#4a3bc9', 'Target': '#1f991f',
                'Others': '#5e5e66', 'Title': '#4b4c53', 'Imprint': '#2e2f37',
                '1': '#fff5f0', '2': '#fee0d2', '3': '#fcbba1',
                '4': '#fc9272', '5': '#fb6a4a', '6': '#ef3b2c',
                '7': '#cb181d', '8': '#a50f15', '9': '#67000d'
            }
            unique_aois = sorted(list(set(sequence)))
            fallback_palette = sns.color_palette("pastel", len(unique_aois)).as_hex()
            aoi_color_map = {aoi: AOI_COLORS.get(aoi, fallback_palette[i % len(fallback_palette)]) for i, aoi in enumerate(unique_aois)}

        results = []
        for end_idx in range(min_w, N + 1):
            if is_xy_mode:
                window_coords = coords[:end_idx]
                if len(window_coords) < 2:
                    metrics = {'RR': 0.0, 'DET': 0.0, 'LAM': 0.0}
                else:
                    dist_matrix = cdist(window_coords, window_coords, metric='euclidean')
                    RP = (dist_matrix <= current_radius).astype(int)
                    np.fill_diagonal(RP, 0)
                    metrics = calculate_rqa_from_matrix(RP, len(window_coords))
            else:
                window_seq = sequence[:end_idx]
                seq_arr = np.array(window_seq)
                if len(seq_arr) < 2:
                    metrics = {'RR': 0.0, 'DET': 0.0, 'LAM': 0.0}
                else:
                    RP = (seq_arr[:, None] == seq_arr[None, :]).astype(int)
                    np.fill_diagonal(RP, 0)
                    metrics = calculate_rqa_from_matrix(RP, len(seq_arr))

            results.append({
                'Fixation_Index': end_idx,
                'RR': metrics['RR'],
                'DET': metrics['DET'],
                'LAM': metrics['LAM']
            })

        df_results = pd.DataFrame(results)

        last_row = df_results.iloc[-1].copy()
        last_row['Fixation_Index'] = N + 1
        df_results = pd.concat([df_results, pd.DataFrame([last_row])], ignore_index=True)

        fig, axes = plt.subplots(3, 1, figsize=(14, 10))
        color_line = '#131314'

        max_rr = df_results['RR'].max()
        max_det = df_results['DET'].max()
        max_lam = df_results['LAM'].max()

        def annotate_jumps(ax, col_name):
            for idx in range(1, len(df_results) - 1):
                diff = df_results[col_name].iloc[idx] - df_results[col_name].iloc[idx-1]
                if abs(diff) >= threshold:
                    x_pos = df_results['Fixation_Index'].iloc[idx]
                    y_pos = df_results[col_name].iloc[idx]
                    if diff > 0:
                        c, y_offset, va_align = '#1a9641', 4, 'bottom'
                        text_val = f"+{diff:.2f}"
                    else:
                        c, y_offset, va_align = '#d7191c', -4, 'top'
                        text_val = f"{diff:.2f}"

                    txt = ax.annotate(text_val, xy=(x_pos, y_pos), xytext=(4, y_offset),
                                      textcoords="offset points", color=c, fontsize=9,
                                      fontweight='bold', va=va_align, ha='left', zorder=15)
                    txt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='white', alpha=0.9)])

        mode_title = "Přístup XY" if is_xy_mode else "Přístup AOI"

        # Graf 1: RR (Využití clip_on=False a zorder=10 pro překrytí spodního rámečku osy)
        axes[0].step(df_results['Fixation_Index'], df_results['RR'], where='post', color=color_line, linewidth=3.5, zorder=10, clip_on=False)
        axes[0].set_title(f"Vývoj metrik v čase ({mode_title})\nRespondent: {part} ({exp}) | Stimul: {stim}", fontsize=15, fontweight='bold', pad=15)
        axes[0].set_ylabel("Recurrence (RR)", fontsize=11, fontweight='bold')
        axes[0].set_xlabel("Pořadí fixací", fontsize=12, fontweight='bold')
        axes[0].set_ylim(0, min(1.05, max(0.1, max_rr + 0.1)))
        annotate_jumps(axes[0], 'RR')

        # Graf 2: DET
        axes[1].step(df_results['Fixation_Index'], df_results['DET'], where='post', color=color_line, linewidth=3.5, zorder=10, clip_on=False)
        axes[1].set_ylabel("Determinismus (DET)", fontsize=11, fontweight='bold')
        axes[1].set_xlabel("Pořadí fixací", fontsize=12, fontweight='bold')
        axes[1].set_ylim(0, min(1.05, max(0.1, max_det + 0.1)))
        annotate_jumps(axes[1], 'DET')

        # Graf 3: LAM
        axes[2].step(df_results['Fixation_Index'], df_results['LAM'], where='post', color=color_line, linewidth=3.5, zorder=10, clip_on=False)
        axes[2].set_ylabel("Laminarita (LAM)", fontsize=11, fontweight='bold')
        axes[2].set_xlabel("Pořadí fixací", fontsize=12, fontweight='bold')
        axes[2].set_ylim(0, min(1.05, max(0.1, max_lam + 0.1)))
        annotate_jumps(axes[2], 'LAM')

        for ax, metric in zip(axes, ['RR', 'DET', 'LAM']):
            ax.annotate(f"{df_results[metric].iloc[-1]:.3f}", xy=(N + 1, df_results[metric].iloc[-1]), xytext=(8, 0),
                        textcoords='offset points', ha='left', va='center', color=color_line, fontweight='bold', fontsize=11, annotation_clip=False)

            ax.set_xlim(min_w, N + 1)
            ax.grid(True, linestyle=':', alpha=0.4, axis='y')

            if not is_xy_mode:
                for i, aoi in enumerate(sequence):
                    if i + 1 >= min_w:
                        ax.axvspan(i + 1, i + 2, color=aoi_color_map[aoi], alpha=0.2, lw=0, zorder=1)

            for i in range(min_w, N + 2):
                ax.axvline(x=i, color='white', linewidth=1.5, zorder=2)

            xticks_pos = np.arange(min_w, N + 1) + 0.5
            xticks_labels = [str(i) for i in range(min_w, N + 1)]
            ax.set_xticks(xticks_pos)
            ax.set_xticklabels(xticks_labels, fontsize=10)

            if N - min_w > 25: ax.tick_params(axis='x', rotation=90)

            ax.tick_params(axis='x', length=0, labelbottom=True)

        if not is_xy_mode:
            legend_patches = [mpatches.Patch(color=color, alpha=0.3, label=aoi) for aoi, color in aoi_color_map.items()]
            axes[1].legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.08, 1), title="AOI", title_fontsize=12)

        plt.tight_layout()
        plt.show()

btn_run.on_click(on_run_clicked)

if not df_cleaned.empty:
    display(widgets.VBox([
        lbl_running,
        dd_rqa_mode,
        widgets.HBox([dd_stimulus, dd_participant]),
        widgets.HBox([min_window_slider, jump_threshold_slider]),
        btn_run,
        out_run
    ]))
else:
    print("Graficke rozhrani se nenacetlo kvuli chybejicim datum.")

In [ ]:
# @title Kvalitativní porovnání časové dynamiky všech respondentů (Heatmap Strip) { display-mode: "form" }
# @markdown Tento modul vygeneruje teplotní mapu (heatmap) pro vybraný stimul a metriku.
# @markdown * **Skupina:** Umožňuje vizualizovat všechny respondenty, nebo vyfiltrovat pouze specifickou zkušenostní skupinu.
# @markdown * **Min fixací:** Pokud má respondent za celý pokus méně fixací, než je toto číslo, bude z grafu zcela vyřazen (např. ignorování odfláknutých pokusů).
# @markdown * **Max fixací:** Ořízne dlouhé záznamy na tuto hodnotu, aby se graf nedeformoval do šířky.

# Ověření dostupnosti dat
try:
    folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    file_cleaned = FILE_COMPLETE_CSV if 'FILE_COMPLETE_CSV' in globals() else folder_path + 'SegmentedData-cleaned.csv'
    df_cleaned = pd.read_csv(file_cleaned, low_memory=False)
except Exception as e:
    print("Data nebyla nalezena. Ujistěte se, že máte spuštěné předchozí načítací kroky.")
    df_cleaned = pd.DataFrame()

if not df_cleaned.empty:
    available_stimuli_hm = sorted(df_cleaned['stimulus'].dropna().unique().tolist())
    available_metrics_hm = ['DET', 'LAM', 'RR']

    # UI Prvky - Horní řada
    dd_stim_hm = widgets.Dropdown(options=available_stimuli_hm, description='Stimul:', layout=widgets.Layout(width='280px'))
    dd_metric_hm = widgets.Dropdown(options=available_metrics_hm, value='LAM', description='Metrika:', layout=widgets.Layout(width='180px'))
    dd_mode_hm = widgets.Dropdown(options=['AOI', 'XY'], value='AOI', description='Přístup:', layout=widgets.Layout(width='180px'))
    dd_group_hm = widgets.Dropdown(options=['Všichni', 'Jen Nováčci', 'Jen Experti'], value='Všichni', description='Skupina:', layout=widgets.Layout(width='220px'))

    # UI Prvky - Spodní řada (limity)
    w_min_fix = widgets.IntText(value=10, description='Min fixací (vyřadit krátké):', style={'description_width': 'initial'}, layout=widgets.Layout(width='250px'))
    w_max_fix = widgets.IntText(value=80, description='Max fixací (oříznout dlouhé):', style={'description_width': 'initial'}, layout=widgets.Layout(width='250px'))

    btn_hm = widgets.Button(description='Vypočítat a vykreslit Heatmap Strip', button_style='primary', layout=widgets.Layout(width='300px', margin='15px 0 0 0'))
    out_hm = widgets.Output()

    # Matematické funkce pro RQA
    def get_line_lengths(arr, min_length=2):
        pad = np.pad(arr, (1, 1), mode='constant', constant_values=0)
        diffs = np.diff(pad)
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0]
        lengths = ends - starts
        return lengths[lengths >= min_length]

    def extract_rqa_from_rp(RP, N, l_min, v_min):
        recurrent_points = np.sum(RP)
        total_possible = N * (N - 1)
        RR = recurrent_points / total_possible if total_possible > 0 else 0
        diag_lines = []
        for d in range(1, N):
            diag = np.diagonal(RP, offset=d)
            diag_lines.extend(get_line_lengths(diag, l_min))
        DET = np.sum(diag_lines) / (recurrent_points / 2) if recurrent_points > 0 and diag_lines else 0
        vert_lines = []
        for c in range(N):
            vert_lines.extend(get_line_lengths(RP[:, c], v_min))
        LAM = np.sum(vert_lines) / recurrent_points if recurrent_points > 0 and vert_lines else 0
        return RR, DET, LAM

    def plot_heatmap_strip(b):
        with out_hm:
            clear_output(wait=True)
            print("Probíhá dynamický výpočet Running RQA pro zvolené respondenty... (Může to několik vteřin trvat)")

            stim = dd_stim_hm.value
            metric = dd_metric_hm.value
            mode = dd_mode_hm.value
            group_filter = dd_group_hm.value

            # Načtení limitů z UI
            min_limit = w_min_fix.value
            max_limit = w_max_fix.value
            min_window = 1

            df_sub = df_cleaned[df_cleaned['stimulus'] == stim].copy()

            # APLIKACE FILTRU ZKUŠENOSTÍ
            if group_filter == 'Jen Nováčci':
                df_sub = df_sub[df_sub['Zkusenosti'] == 'Nováček']
            elif group_filter == 'Jen Experti':
                df_sub = df_sub[df_sub['Zkusenosti'] == 'Expert']

            if df_sub.empty:
                print("Pro tuto volbu (stimul + skupina) nejsou k dispozici žádná data.")
                return

            participants = sorted(df_sub['participant'].unique())
            all_results = []

            # Poloměr pro XY
            if mode == 'XY':
                file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
                try:
                    df_rqa_xy = pd.read_csv(file_rqa_xy)
                    radius = df_rqa_xy['Radius_px'].iloc[0] if 'Radius_px' in df_rqa_xy.columns else 50
                except:
                    radius = 50

            # Hromadný výpočet Running RQA
            for part in participants:
                subset = df_sub[df_sub['participant'] == part].sort_values('timestamp').copy()
                exp = subset['Zkusenosti'].iloc[0] if 'Zkusenosti' in subset.columns else 'Neznámo'
                N = len(subset)

                if N < min_limit:
                    continue

                actual_limit = min(N, max_limit)

                for w in range(min_window, actual_limit + 1):
                    win_df = subset.iloc[:w]
                    N_w = len(win_df)

                    if mode == 'AOI':
                        seq = np.array(win_df['AOI Single'].astype(str).tolist())
                        RP = (seq[:, None] == seq[None, :]).astype(int)
                    else:
                        coords = np.column_stack((pd.to_numeric(win_df['x'], errors='coerce'), pd.to_numeric(win_df['y'], errors='coerce')))
                        dist_matrix = cdist(coords, coords, metric='euclidean')
                        RP = (dist_matrix <= radius).astype(int)

                    np.fill_diagonal(RP, 0)
                    rr, det, lam = extract_rqa_from_rp(RP, N_w, 2, 2)

                    all_results.append({
                        'participant': part,
                        'Zkusenosti': exp,
                        'fixation_idx': w,
                        'RR': rr,
                        'DET': det,
                        'LAM': lam
                    })

            if not all_results:
                print(f"Nelze vypočítat - žádný respondent v této skupině neměl více než {min_limit} fixací.")
                return

            df_running = pd.DataFrame(all_results)

            # Příprava popisků
            df_running['label'] = df_running['Zkusenosti'].astype(str) + ' | ' + df_running['participant'].astype(str)
            pivot_df = df_running.pivot_table(index='label', columns='fixation_idx', values=metric)

            # Správné seřazení Y-osy
            sorted_labels = df_running[['Zkusenosti', 'label']].drop_duplicates().sort_values(['Zkusenosti', 'label'])['label'].tolist()
            pivot_df = pivot_df.reindex(sorted_labels)

            # Vykreslení - proměnlivá výčka grafu
            fig, ax = plt.subplots(figsize=(16, max(4, len(sorted_labels) * 0.35)))
            cmap_choice = 'Reds'

            # Jemnější a světlejší mřížka
            sns.heatmap(pivot_df, cmap=cmap_choice, ax=ax,
                        cbar_kws={'label': f'Hodnota {metric}', 'shrink': 0.8, 'pad': 0.02},
                        linewidths=0.3, linecolor='lightgray',
                        xticklabels=False, yticklabels=True, vmin=0, vmax=1)

            title_text = f'Heatmap Strip: Časová dynamika {metric}\n(Stimul: {stim}, Přístup: {mode}, Skupina: {group_filter})'
            ax.set_title(title_text, fontsize=16, fontweight='bold', pad=20)
            ax.set_xlabel(f'Pořadí fixace (zobrazeno do max. {max_limit} fixací)', fontsize=13, fontweight='bold', labelpad=15)
            ax.set_ylabel('Respondent (seskupeno podle zkušenosti)', fontsize=13, fontweight='bold', labelpad=10)

            # Vlastní výpočet pozic pro čistou osu X
            col_vals = pivot_df.columns.tolist()
            tick_positions = []
            tick_labels = []

            for i, val in enumerate(col_vals):
                if val == 1 or val % 5 == 0:
                    tick_positions.append(i + 0.5)
                    tick_labels.append(str(val))

            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=0, fontsize=11)
            ax.tick_params(axis='y', labelsize=11)

            plt.tight_layout()
            plt.show()

    btn_hm.on_click(plot_heatmap_strip)
    ui_hm = widgets.VBox([
        widgets.HBox([dd_stim_hm, dd_mode_hm, dd_metric_hm, dd_group_hm]),
        widgets.HBox([w_min_fix, w_max_fix]),
        btn_hm
    ])
    display(ui_hm, out_hm)

In [ ]:
# @title Kvalitativní porovnání časové dynamiky všech respondentů (Heatmap Strip) { display-mode: "form" }
# @markdown Tento modul vygeneruje teplotní mapu (heatmap) pro vybraný stimul a metriku.
# @markdown * **Skupina:** Umožňuje vizualizovat všechny, vyfiltrovat Nováčky/Experty, nebo rozdělit respondenty podle toho, zda úlohu vyřešili rychle (pod mediánem) nebo pomalu (nad mediánem).
# @markdown * **Min/Max fixací:** Omezení délky záznamu pro čistší vizualizaci.

import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

# Ověření dostupnosti dat
try:
    folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    file_cleaned = FILE_COMPLETE_CSV if 'FILE_COMPLETE_CSV' in globals() else folder_path + 'SegmentedData-cleaned.csv'
    df_cleaned = pd.read_csv(file_cleaned, low_memory=False)
except Exception as e:
    print("Data nebyla nalezena. Ujistěte se, že máte spuštěné předchozí načítací kroky.")
    df_cleaned = pd.DataFrame()

if not df_cleaned.empty:
    available_stimuli_hm = sorted(df_cleaned['stimulus'].dropna().unique().tolist())
    available_metrics_hm = ['DET', 'LAM', 'RR']

    # UI Prvky - Horní řada
    dd_stim_hm = widgets.Dropdown(options=available_stimuli_hm, description='Stimul:', layout=widgets.Layout(width='280px'))
    dd_metric_hm = widgets.Dropdown(options=available_metrics_hm, value='LAM', description='Metrika:', layout=widgets.Layout(width='180px'))
    dd_mode_hm = widgets.Dropdown(options=['AOI', 'XY'], value='AOI', description='Přístup:', layout=widgets.Layout(width='180px'))

    # NOVÉ MOŽNOSTI VE VÝBĚRU SKUPINY
    group_options = [
        'Všichni',
        'Jen Nováčci',
        'Jen Experti',
        'Pod mediánem fixací (Rychlí)',
        'Nad mediánem fixací (Pomalí)',
        'Srovnání: Rychlí vs. Pomalí' # <-- Přidána možnost pro zobrazení nad sebou
    ]
    dd_group_hm = widgets.Dropdown(options=group_options, value='Všichni', description='Skupina:', layout=widgets.Layout(width='300px'))

    # UI Prvky - Spodní řada (limity)
    w_min_fix = widgets.IntText(value=10, description='Min fixací (vyřadit krátké):', style={'description_width': 'initial'}, layout=widgets.Layout(width='250px'))
    w_max_fix = widgets.IntText(value=80, description='Max fixací (oříznout dlouhé):', style={'description_width': 'initial'}, layout=widgets.Layout(width='250px'))

    btn_hm = widgets.Button(description='Vypočítat a vykreslit Heatmap Strip', button_style='primary', layout=widgets.Layout(width='300px', margin='15px 0 0 0'))
    out_hm = widgets.Output()

    # Matematické funkce pro RQA
    def get_line_lengths(arr, min_length=2):
        pad = np.pad(arr, (1, 1), mode='constant', constant_values=0)
        diffs = np.diff(pad)
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0]
        lengths = ends - starts
        return lengths[lengths >= min_length]

    def extract_rqa_from_rp(RP, N, l_min, v_min):
        recurrent_points = np.sum(RP)
        total_possible = N * (N - 1)
        RR = recurrent_points / total_possible if total_possible > 0 else 0
        diag_lines = []
        for d in range(1, N):
            diag = np.diagonal(RP, offset=d)
            diag_lines.extend(get_line_lengths(diag, l_min))
        DET = np.sum(diag_lines) / (recurrent_points / 2) if recurrent_points > 0 and diag_lines else 0
        vert_lines = []
        for c in range(N):
            vert_lines.extend(get_line_lengths(RP[:, c], v_min))
        LAM = np.sum(vert_lines) / recurrent_points if recurrent_points > 0 and vert_lines else 0
        return RR, DET, LAM

    # Pomocná funkce pro jednotné vykreslení heatmapy
    def draw_single_heatmap(ax, pivot_df, title, metric):
        sns.heatmap(pivot_df, cmap='Reds', ax=ax,
                    cbar_kws={'label': f'Hodnota {metric}', 'shrink': 0.8, 'pad': 0.02},
                    linewidths=0.3, linecolor='lightgray',
                    xticklabels=False, yticklabels=True, vmin=0, vmax=1)

        ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
        ax.set_ylabel('Respondent (Zkušenost | ID)', fontsize=11, fontweight='bold', labelpad=10)

        # Vlastní výpočet pozic pro čistou osu X
        col_vals = pivot_df.columns.tolist()
        tick_positions = []
        tick_labels = []
        for i, val in enumerate(col_vals):
            if val == 1 or val % 5 == 0:
                tick_positions.append(i + 0.5)
                tick_labels.append(str(val))

        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=0, fontsize=11)
        ax.tick_params(axis='y', labelsize=10)


    def plot_heatmap_strip(b):
        with out_hm:
            clear_output(wait=True)
            print("Probíhá dynamický výpočet Running RQA pro zvolené respondenty... (Může to několik vteřin trvat)")

            stim = dd_stim_hm.value
            metric = dd_metric_hm.value
            mode = dd_mode_hm.value
            group_filter = dd_group_hm.value

            min_limit = w_min_fix.value
            max_limit = w_max_fix.value
            min_window = 1

            df_sub = df_cleaned[df_cleaned['stimulus'] == stim].copy()

            if df_sub.empty:
                print("Pro tuto volbu (stimul) nejsou k dispozici žádná data.")
                return

            # Výpočet mediánu fixací pro daný stimul
            fix_counts = df_sub.groupby('participant').size()
            median_fix = fix_counts.median()

            # APLIKACE FILTRU
            if group_filter == 'Jen Nováčci':
                df_sub = df_sub[df_sub['Zkusenosti'] == 'Nováček']
            elif group_filter == 'Jen Experti':
                df_sub = df_sub[df_sub['Zkusenosti'] == 'Expert']
            elif group_filter == 'Pod mediánem fixací (Rychlí)':
                df_sub = df_sub[df_sub['participant'].isin(fix_counts[fix_counts <= median_fix].index)]
            elif group_filter == 'Nad mediánem fixací (Pomalí)':
                df_sub = df_sub[df_sub['participant'].isin(fix_counts[fix_counts > median_fix].index)]
            # Pro srovnání necháváme všechny

            if df_sub.empty:
                print("Pro tuto volbu filtru nezbyla žádná data.")
                return

            participants = sorted(df_sub['participant'].unique())
            all_results = []

            if mode == 'XY':
                file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
                try:
                    df_rqa_xy = pd.read_csv(file_rqa_xy)
                    radius = df_rqa_xy['Radius_px'].iloc[0] if 'Radius_px' in df_rqa_xy.columns else 50
                except:
                    radius = 50

            for part in participants:
                subset = df_sub[df_sub['participant'] == part].sort_values('timestamp').copy()
                exp = subset['Zkusenosti'].iloc[0] if 'Zkusenosti' in subset.columns else 'Neznámo'
                N = len(subset)

                if N < min_limit:
                    continue

                # Určení kategorie rychlosti pro daného participanta
                speed_group = 'Rychlí' if fix_counts[part] <= median_fix else 'Pomalí'

                actual_limit = min(N, max_limit)

                for w in range(min_window, actual_limit + 1):
                    win_df = subset.iloc[:w]
                    N_w = len(win_df)

                    if mode == 'AOI':
                        seq = np.array(win_df['AOI Single'].astype(str).tolist())
                        RP = (seq[:, None] == seq[None, :]).astype(int)
                    else:
                        coords = np.column_stack((pd.to_numeric(win_df['x'], errors='coerce'), pd.to_numeric(win_df['y'], errors='coerce')))
                        dist_matrix = cdist(coords, coords, metric='euclidean')
                        RP = (dist_matrix <= radius).astype(int)

                    np.fill_diagonal(RP, 0)
                    rr, det, lam = extract_rqa_from_rp(RP, N_w, 2, 2)

                    all_results.append({
                        'participant': part,
                        'Zkusenosti': exp,
                        'SpeedGroup': speed_group,
                        'fixation_idx': w,
                        'RR': rr,
                        'DET': det,
                        'LAM': lam
                    })

            if not all_results:
                print(f"Nelze vypočítat - žádný respondent nesplňuje minimální počet {min_limit} fixací.")
                return

            df_running = pd.DataFrame(all_results)
            df_running['label'] = df_running['Zkusenosti'].astype(str) + ' | ' + df_running['participant'].astype(str)

            # VYKRESLENÍ: Rozdělení podle zvoleného zobrazení
            if group_filter == 'Srovnání: Rychlí vs. Pomalí':
                print(f"INFO: Medián pro stimul {stim} je {median_fix:.0f} fixací. Generuji srovnávací pohled.")

                df_fast = df_running[df_running['SpeedGroup'] == 'Rychlí']
                df_slow = df_running[df_running['SpeedGroup'] == 'Pomalí']

                # Zabráníme pádu, pokud jedna ze skupin zůstala po filtru 'min_limit' prázdná
                if df_fast.empty or df_slow.empty:
                    print("POZOR: Jedna ze skupin nemá dostatek dat pro vykreslení (patrně kvůli limitu 'Min fixací'). Zkuste limit snížit.")
                    return

                pivot_fast = df_fast.pivot_table(index='label', columns='fixation_idx', values=metric)
                pivot_slow = df_slow.pivot_table(index='label', columns='fixation_idx', values=metric)

                sorted_labels_fast = df_fast[['Zkusenosti', 'label']].drop_duplicates().sort_values(['Zkusenosti', 'label'])['label'].tolist()
                sorted_labels_slow = df_slow[['Zkusenosti', 'label']].drop_duplicates().sort_values(['Zkusenosti', 'label'])['label'].tolist()

                pivot_fast = pivot_fast.reindex(sorted_labels_fast)
                pivot_slow = pivot_slow.reindex(sorted_labels_slow)

                # Dva grafy pod sebou
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, max(8, (len(pivot_fast) + len(pivot_slow)) * 0.35)))

                fig.suptitle(f'Srovnávací Heatmap Strip: Časová dynamika {metric} (Stimul: {stim})\nRozděleno podle celkového kognitivního úsilí (Medián = {median_fix:.0f} fixací)', fontsize=16, fontweight='bold', y=1.02)

                draw_single_heatmap(ax1, pivot_fast, f'Skupina: Rychlí řešitelé (Pod mediánem)', metric)
                draw_single_heatmap(ax2, pivot_slow, f'Skupina: Pomalí řešitelé (Nad mediánem)', metric)

                ax2.set_xlabel(f'Pořadí fixace (zobrazeno do max. {max_limit} fixací)', fontsize=13, fontweight='bold', labelpad=15)

            else:
                pivot_df = df_running.pivot_table(index='label', columns='fixation_idx', values=metric)
                sorted_labels = df_running[['Zkusenosti', 'label']].drop_duplicates().sort_values(['Zkusenosti', 'label'])['label'].tolist()
                pivot_df = pivot_df.reindex(sorted_labels)

                fig, ax = plt.subplots(figsize=(16, max(4, len(sorted_labels) * 0.35)))
                title_text = f'Heatmap Strip: Časová dynamika {metric}\n(Stimul: {stim}, Přístup: {mode}, Skupina: {group_filter})'

                draw_single_heatmap(ax, pivot_df, title_text, metric)
                ax.set_xlabel(f'Pořadí fixace (zobrazeno do max. {max_limit} fixací)', fontsize=13, fontweight='bold', labelpad=15)

            plt.tight_layout()
            plt.show()

    btn_hm.on_click(plot_heatmap_strip)
    ui_hm = widgets.VBox([
        widgets.HBox([dd_stim_hm, dd_mode_hm, dd_metric_hm, dd_group_hm]),
        widgets.HBox([w_min_fix, w_max_fix]),
        btn_hm
    ])
    display(ui_hm, out_hm)

In [ ]:
# @title Individuální profil respondenta: Časová dynamika napříč stimuly { display-mode: "form" }
# @markdown Tento modul vygeneruje "pás karet" (heatmap strips) pro jednoho konkrétního respondenta.
# @markdown Uvidíte pod sebou jeho vizuální strategii pro všechny mapy, které během testování viděl.



# Ověření dostupnosti dat
try:
    folder_path = PROJECT_FOLDER if 'PROJECT_FOLDER' in globals() else 'drive/MyDrive/BAK-RQA/'
    file_cleaned = FILE_COMPLETE_CSV if 'FILE_COMPLETE_CSV' in globals() else folder_path + 'SegmentedData-cleaned.csv'
    df_cleaned = pd.read_csv(file_cleaned, low_memory=False)
except Exception as e:
    print("Data nebyla nalezena. Ujistěte se, že máte spuštěné předchozí načítací kroky.")
    df_cleaned = pd.DataFrame()

if not df_cleaned.empty:
    available_participants = sorted(df_cleaned['participant'].unique().tolist())
    available_metrics_profile = ['DET', 'LAM', 'RR']

    # UI Prvky
    dd_part_profile = widgets.Dropdown(options=available_participants, description='Respondent:', layout=widgets.Layout(width='250px'))
    dd_metric_profile = widgets.Dropdown(options=available_metrics_profile, value='LAM', description='Metrika:', layout=widgets.Layout(width='180px'))
    dd_mode_profile = widgets.Dropdown(options=['AOI', 'XY'], value='AOI', description='Přístup:', layout=widgets.Layout(width='180px'))
    w_max_fix_profile = widgets.IntText(value=80, description='Max fixací (osa X):', style={'description_width': 'initial'}, layout=widgets.Layout(width='200px'))

    btn_profile = widgets.Button(description='Vygenerovat individuální profil', button_style='success', layout=widgets.Layout(width='300px', margin='10px 0 0 0'))
    out_profile = widgets.Output()

    def get_line_lengths(arr, min_length=2):
        pad = np.pad(arr, (1, 1), mode='constant', constant_values=0)
        diffs = np.diff(pad)
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0]
        lengths = ends - starts
        return lengths[lengths >= min_length]

    def extract_rqa_from_rp(RP, N, l_min, v_min):
        recurrent_points = np.sum(RP)
        total_possible = N * (N - 1)
        RR = recurrent_points / total_possible if total_possible > 0 else 0
        diag_lines = []
        for d in range(1, N):
            diag = np.diagonal(RP, offset=d)
            diag_lines.extend(get_line_lengths(diag, l_min))
        DET = np.sum(diag_lines) / (recurrent_points / 2) if recurrent_points > 0 and diag_lines else 0
        vert_lines = []
        for c in range(N):
            vert_lines.extend(get_line_lengths(RP[:, c], v_min))
        LAM = np.sum(vert_lines) / recurrent_points if recurrent_points > 0 and vert_lines else 0
        return RR, DET, LAM

    def draw_profile(b):
        with out_profile:
            clear_output(wait=True)
            part_id = dd_part_profile.value
            metric = dd_metric_profile.value
            mode = dd_mode_profile.value
            max_fix = w_max_fix_profile.value

            df_p = df_cleaned[df_cleaned['participant'] == part_id].copy()
            stimuli = sorted(df_p['stimulus'].unique().tolist())

            exp_info = df_p['Zkusenosti'].iloc[0] if 'Zkusenosti' in df_p.columns else 'Neznámo'
            print(f"Generuji profil pro: {part_id} ({exp_info})")
            print(f"Počet nalezených stimulů: {len(stimuli)}")

            all_data = []

            # Poloměr pro XY
            radius = 50
            if mode == 'XY':
                try:
                    file_rqa_xy = FILE_OUT_XY if 'FILE_OUT_XY' in globals() else folder_path + 'rqa_results_XY.csv'
                    radius = pd.read_csv(file_rqa_xy)['Radius_px'].iloc[0]
                except: pass

            for stim in stimuli:
                subset = df_p[df_p['stimulus'] == stim].sort_values('timestamp').copy()
                N = len(subset)
                actual_limit = min(N, max_fix)

                for w in range(1, actual_limit + 1):
                    win_df = subset.iloc[:w]
                    N_w = len(win_df)
                    if mode == 'AOI':
                        seq = np.array(win_df['AOI Single'].astype(str).tolist())
                        RP = (seq[:, None] == seq[None, :]).astype(int)
                    else:
                        coords = np.column_stack((pd.to_numeric(win_df['x'], errors='coerce'), pd.to_numeric(win_df['y'], errors='coerce')))
                        dist_matrix = cdist(coords, coords, metric='euclidean')
                        RP = (dist_matrix <= radius).astype(int)

                    np.fill_diagonal(RP, 0)
                    rr, det, lam = extract_rqa_from_rp(RP, N_w, 2, 2)
                    all_data.append({'stimulus': stim, 'fixation_idx': w, 'RR': rr, 'DET': det, 'LAM': lam})

            if not all_data:
                print("Chyba při výpočtu dat.")
                return

            df_res = pd.DataFrame(all_data)
            pivot_res = df_res.pivot_table(index='stimulus', columns='fixation_idx', values=metric)

            # Vykreslení
            fig, ax = plt.subplots(figsize=(16, len(stimuli) * 0.8 + 2))
            sns.heatmap(pivot_res, cmap='Reds', ax=ax, linewidths=0.2, linecolor='lightgray',
                        cbar_kws={'label': f'Hodnota {metric}', 'shrink': 0.5}, vmin=0, vmax=1)

            ax.set_title(f'Individuální RQA profil: Respondent {part_id} ({exp_info})\nMetrika: {metric}, Režim: {mode}',
                         fontsize=16, fontweight='bold', pad=20)
            ax.set_xlabel('Pořadí fixace', fontsize=12, fontweight='bold')
            ax.set_ylabel('Stimul (Mapa)', fontsize=12, fontweight='bold')

            # Úprava ticků osy X
            col_vals = pivot_res.columns.tolist()
            ax.set_xticks([i + 0.5 for i, v in enumerate(col_vals) if v == 1 or v % 5 == 0])
            ax.set_xticklabels([int(v) for v in col_vals if v == 1 or v % 5 == 0], rotation=0)

            plt.tight_layout()
            plt.show()

    btn_profile.on_click(draw_profile)
    display(widgets.VBox([
        widgets.HBox([dd_part_profile, dd_metric_profile, dd_mode_profile, w_max_fix_profile]),
        btn_profile, out_profile
    ]))